In [1]:
import os
import gzip
import hashlib
import math
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import requests

from scipy import stats
from scipy.sparse.linalg import splu

# -----------------------------------------------------------------------------
# GOOGLE DRIVE
# -----------------------------------------------------------------------------

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

BASE_DIR = Path("/content/drive/MyDrive/Markov_Antifragility_IEEE")
DATA_DIR = BASE_DIR / "data"
RESULTS_DIR = BASE_DIR / "results_real_only"
FIGS_DIR = RESULTS_DIR / "figures"

for directory in (DATA_DIR, RESULTS_DIR, FIGS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# -----------------------------------------------------------------------------
# EXPERIMENT CONFIGURATION
# -----------------------------------------------------------------------------

MASTER_SEED = 20260813

# Nine progressively increasing damage conditions. f=0 is the separate baseline.
DAMAGE_FRACS = np.array([0.05, 0.10, 0.15, 0.20, 0.25,
                         0.30, 0.35, 0.40, 0.45], dtype=float)

# A descriptive improvement must exceed numerical triviality.
IMPROVEMENT_THRESHOLD = 0.98       # R(f) < 0.98 => at least 2% improvement

# Hutchinson probes used to estimate trace(L^+) for average hitting time.
TRACE_PROBES = 24

# Real-data independent progressive damage trajectories.
# These are experimental settings, NOT data values.
TRIALS = {
    "ca_GrQc": 20,
    "ca_HepTh": 15,
    "Facebook_Combined": 20,
    "Email_Eu_Core": 30,
    "Wiki_Vote": 20,
    "CollegeMsg": 30,
}

# LCC-size controls at representative damage levels.
SIZE_CONTROL_FRACS = {0.10, 0.20, 0.30, 0.40}

# -----------------------------------------------------------------------------
# PUBLIC REAL DATASETS — SOURCE URLS ONLY
# -----------------------------------------------------------------------------

DATASETS = {
    "ca_GrQc": {
        "url": "https://snap.stanford.edu/data/ca-GrQc.txt.gz",
        "filename": "ca-GrQc.txt.gz",
        "columns": 2,
        "source_type": "undirected",
        "transform": "simple undirected graph; remove self-loops; retain baseline LCC",
    },
    "ca_HepTh": {
        "url": "https://snap.stanford.edu/data/ca-HepTh.txt.gz",
        "filename": "ca-HepTh.txt.gz",
        "columns": 2,
        "source_type": "undirected",
        "transform": "simple undirected graph; remove self-loops; retain baseline LCC",
    },
    "Facebook_Combined": {
        "url": "https://snap.stanford.edu/data/facebook_combined.txt.gz",
        "filename": "facebook_combined.txt.gz",
        "columns": 2,
        "source_type": "undirected",
        "transform": "simple undirected graph; remove self-loops; retain baseline LCC",
    },
    "Email_Eu_Core": {
        "url": "https://snap.stanford.edu/data/email-Eu-core.txt.gz",
        "filename": "email-Eu-core.txt.gz",
        "columns": 2,
        "source_type": "directed",
        "transform": "observed directed pairs collapsed to simple undirected graph; retain baseline LCC",
    },
    "Wiki_Vote": {
        "url": "https://snap.stanford.edu/data/wiki-Vote.txt.gz",
        "filename": "wiki-Vote.txt.gz",
        "columns": 2,
        "source_type": "directed",
        "transform": "observed directed vote pairs collapsed to simple undirected graph; retain baseline LCC",
    },
    "CollegeMsg": {
        "url": "https://snap.stanford.edu/data/CollegeMsg.txt.gz",
        "filename": "CollegeMsg.txt.gz",
        "columns": 3,
        "source_type": "directed_temporal",
        "transform": "observed message pairs collapsed over time to simple undirected graph; retain baseline LCC",
    },
}

# -----------------------------------------------------------------------------
# REPRODUCIBILITY
# -----------------------------------------------------------------------------

def stable_seed(*parts):
    """Deterministic uint32 seed from semantic identifiers."""
    payload = "|".join(map(str, parts)).encode("utf-8")
    digest = hashlib.sha256(payload).digest()
    return (int.from_bytes(digest[:4], "little") ^ MASTER_SEED) & 0xFFFFFFFF


def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


# -----------------------------------------------------------------------------
# DOWNLOAD + PARSE REAL DATA
# -----------------------------------------------------------------------------

def download_file(url, path):
    """Stream a public SNAP file to disk. Existing nonempty file is retained."""
    path = Path(path)
    if path.exists() and path.stat().st_size > 0:
        print(f"  existing: {path.name}")
        return

    print(f"  downloading: {url}")
    headers = {"User-Agent": "Mozilla/5.0 MarkovAntifragilityResearch/1.0"}

    with requests.get(url, stream=True, timeout=120, headers=headers) as r:
        r.raise_for_status()
        tmp = path.with_suffix(path.suffix + ".part")
        with open(tmp, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
        tmp.replace(path)

    if not path.exists() or path.stat().st_size == 0:
        raise RuntimeError(f"Download failed or produced empty file: {url}")


def parse_snap_graph(path, columns):
    """
    Read only the first two columns as observed source/destination node IDs.
    Extra columns (e.g., CollegeMsg timestamp) are retained in the raw source
    file but are not needed for the static graph.
    """
    G = nx.Graph()
    raw_edge_rows = 0

    with gzip.open(path, "rt", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue

            parts = line.split()
            if len(parts) < 2:
                continue

            try:
                u = int(parts[0])
                v = int(parts[1])
            except ValueError:
                # SNAP files used here have integer IDs. If a malformed header
                # appears, skip it rather than fabricating an ID.
                continue

            raw_edge_rows += 1

            if u != v:
                G.add_edge(u, v)

    if G.number_of_nodes() == 0 or G.number_of_edges() == 0:
        raise RuntimeError(f"No graph data parsed from {path}")

    return G, raw_edge_rows


def largest_connected_component(G):
    G = nx.Graph(G)
    G.remove_edges_from(nx.selfloop_edges(G))

    if G.number_of_nodes() == 0:
        return G

    if nx.is_connected(G):
        return G.copy()

    nodes = max(nx.connected_components(G), key=len)
    return G.subgraph(nodes).copy()


def load_all_real_networks():
    graphs = {}
    provenance = []

    print("\n" + "=" * 80)
    print("DOWNLOADING / LOADING REAL PUBLIC DATASETS")
    print("=" * 80)

    for name, meta in DATASETS.items():
        print(f"\n{name}")
        local_path = DATA_DIR / meta["filename"]
        download_file(meta["url"], local_path)

        G_observed, raw_rows = parse_snap_graph(local_path, meta["columns"])
        pre_n = G_observed.number_of_nodes()
        pre_m = G_observed.number_of_edges()

        G = largest_connected_component(G_observed)

        provenance.append({
            "graph": name,
            "source_url": meta["url"],
            "local_filename": meta["filename"],
            "sha256": sha256_file(local_path),
            "source_type": meta["source_type"],
            "transformation": meta["transform"],
            "raw_noncomment_edge_rows": raw_rows,
            "simple_graph_nodes_before_lcc": pre_n,
            "simple_graph_edges_before_lcc": pre_m,
            "analysis_nodes_after_lcc": G.number_of_nodes(),
            "analysis_edges_after_lcc": G.number_of_edges(),
        })

        graphs[name] = G

        print(f"  raw observed rows: {raw_rows:,}")
        print(f"  simple graph:      n={pre_n:,}, m={pre_m:,}")
        print(f"  analysis LCC:      n={G.number_of_nodes():,}, m={G.number_of_edges():,}")

    provenance_df = pd.DataFrame(provenance)
    provenance_df.to_csv(RESULTS_DIR / "dataset_provenance.csv", index=False)

    return graphs, provenance_df


# -----------------------------------------------------------------------------
# NETWORK STRUCTURE
# -----------------------------------------------------------------------------

def louvain_structure(G, seed):
    try:
        communities = nx.community.louvain_communities(
            G, seed=seed, weight=None, resolution=1.0
        )
    except Exception:
        communities = list(nx.community.greedy_modularity_communities(G))

    q = nx.community.modularity(G, communities, weight=None, resolution=1.0)
    return float(q), communities


def network_descriptors(G, name):
    deg = np.asarray([d for _, d in G.degree()], dtype=float)
    q, communities = louvain_structure(G, stable_seed(name, "louvain"))

    try:
        assortativity = float(nx.degree_assortativity_coefficient(G))
    except Exception:
        assortativity = np.nan

    return {
        "graph": name,
        "nodes": G.number_of_nodes(),
        "edges": G.number_of_edges(),
        "density": float(nx.density(G)),
        "mean_degree": float(deg.mean()),
        "degree_std": float(deg.std(ddof=1)) if len(deg) > 1 else np.nan,
        "degree_cv": float(deg.std(ddof=1) / deg.mean())
                     if len(deg) > 1 and deg.mean() > 0 else np.nan,
        "average_clustering": float(nx.average_clustering(G)),
        "transitivity": float(nx.transitivity(G)),
        "assortativity": assortativity,
        "modularity": q,
        "communities": len(communities),
    }


# -----------------------------------------------------------------------------
# EXACT METRIC DEFINITION + SCALABLE ESTIMATOR
# -----------------------------------------------------------------------------

def average_hitting_time(G, probes=TRACE_PROBES, seed=0):
    r"""
    Estimate the average ordered-pair simple-random-walk hitting time for a
    connected, undirected graph:

        H_bar = [2m/(n-1)] tr(L^+)

    where L^+ is the Moore-Penrose pseudoinverse of the combinatorial Laplacian.

    To avoid forming L^+, estimate tr(L^+) with a Hutchinson estimator.
    For each Rademacher vector z, project z onto 1^\perp, solve the grounded
    Laplacian system, and evaluate z^T L^+ z.

    Returns
    -------
    hitting_time : float
    estimator_se : float
        Monte Carlo standard error from the trace probes. This is numerical
        estimator uncertainty, NOT the uncertainty used for damage inference.
    """
    G = largest_connected_component(G)

    n = G.number_of_nodes()
    m = G.number_of_edges()

    if n < 2 or m < 1:
        return np.nan, np.nan

    # Deterministic matrix ordering.
    H = nx.convert_node_labels_to_integers(G, ordering="sorted")
    L = nx.laplacian_matrix(H).astype(np.float64).tocsc()

    # Ground last node. For a connected graph, this principal minor is SPD.
    Lg = L[:-1, :-1].tocsc()
    lu = splu(Lg)

    rng = np.random.default_rng(seed)
    q = np.empty(probes, dtype=np.float64)

    for i in range(probes):
        z = rng.choice(np.array([-1.0, 1.0]), size=n)
        z -= z.mean()            # projection onto nullspace-orthogonal subspace

        x_ground = lu.solve(z[:-1])

        # With grounded x_n = 0, the quadratic form is gauge invariant.
        q[i] = float(z[:-1] @ x_ground)

    trace_est = float(q.mean())
    trace_se = float(q.std(ddof=1) / math.sqrt(probes)) if probes > 1 else np.nan

    factor = 2.0 * m / (n - 1.0)
    return factor * trace_est, factor * trace_se


# -----------------------------------------------------------------------------
# PROGRESSIVE EDGE-DAMAGE TRAJECTORIES
# -----------------------------------------------------------------------------

def preferential_edge_order(G, seed):
    """
    Generate ONE weighted-without-replacement edge order per realization.

    Baseline edge importance:
        w(u,v) = deg_G(u) + deg_G(v)

    Exponential-race keys implement a weighted random ordering:
        key_e ~ Exp(rate = w_e)
    and ascending keys are removed first.

    Because the same order is used for all fractions in a trial, 10%, 20%, ...
    damage are truly progressive/nested rather than unrelated fresh samples.
    """
    edges = list(G.edges())
    degree = dict(G.degree())

    weights = np.asarray(
        [degree[u] + degree[v] for u, v in edges],
        dtype=np.float64
    )

    if np.any(weights <= 0):
        raise RuntimeError("Encountered nonpositive preferential damage weight.")

    rng = np.random.default_rng(seed)

    # numpy exponential uses scale = 1/rate.
    keys = rng.exponential(scale=1.0 / weights)
    order = np.argsort(keys)

    return [edges[i] for i in order]


def random_edge_order(G, seed):
    """Uniform random edge permutation for a nested random-damage trajectory."""
    edges = list(G.edges())
    rng = np.random.default_rng(seed)
    order = rng.permutation(len(edges))
    return [edges[i] for i in order]


def damage_from_order(G, ordered_edges, frac):
    H = G.copy()
    k = int(round(float(frac) * G.number_of_edges()))
    if k > 0:
        H.remove_edges_from(ordered_edges[:k])
    return H


# -----------------------------------------------------------------------------
# SIZE-MATCHED CONTROL — SUBGRAPH OF THE SAME REAL NETWORK
# -----------------------------------------------------------------------------

def connected_size_matched_subgraph(G, target_n, seed):
    """
    Select a connected induced subgraph of the ORIGINAL real network with the
    requested node count by randomized frontier growth.

    This is NOT a synthetic graph. Every selected node and every retained edge
    comes from the original observed network.
    """
    G = largest_connected_component(G)
    target_n = int(max(2, min(target_n, G.number_of_nodes())))

    if target_n >= G.number_of_nodes():
        return G.copy()

    rng = np.random.default_rng(seed)
    all_nodes = list(G.nodes())

    start = all_nodes[int(rng.integers(0, len(all_nodes)))]
    selected = {start}
    frontier = set(G.neighbors(start))

    while len(selected) < target_n:
        frontier.difference_update(selected)

        if not frontier:
            # Connected G should normally prevent this. If it happens, choose
            # an observed neighbor frontier by finding a boundary edge.
            boundary = []
            for u in selected:
                boundary.extend(v for v in G.neighbors(u) if v not in selected)
            frontier = set(boundary)

        if not frontier:
            raise RuntimeError("Could not grow a connected size-matched subgraph.")

        candidates = tuple(frontier)
        v = candidates[int(rng.integers(0, len(candidates)))]
        selected.add(v)
        frontier.update(G.neighbors(v))

    H = G.subgraph(selected).copy()

    if H.number_of_nodes() != target_n or not nx.is_connected(H):
        raise RuntimeError("Size-matched connected-subgraph construction failed.")

    return H


# -----------------------------------------------------------------------------
# RESUMABLE RAW-RESULT STORAGE
# -----------------------------------------------------------------------------

RAW_PATH = RESULTS_DIR / "real_networks_raw.csv"
BASELINE_PATH = RESULTS_DIR / "baseline_transport.csv"


def load_completed_raw():
    if not RAW_PATH.exists():
        return pd.DataFrame()

    try:
        df = pd.read_csv(RAW_PATH)
        return df
    except Exception:
        warnings.warn("Could not read existing raw results; starting a new raw table.")
        return pd.DataFrame()


def completed_keys(df):
    if df.empty:
        return set()

    required = {"graph", "trial", "damage_type", "damage_frac"}
    if not required.issubset(df.columns):
        return set()

    return set(
        zip(
            df["graph"].astype(str),
            df["trial"].astype(int),
            df["damage_type"].astype(str),
            df["damage_frac"].round(4)
        )
    )


def append_rows(rows):
    if not rows:
        return

    new = pd.DataFrame(rows)
    header = not RAW_PATH.exists()
    new.to_csv(RAW_PATH, mode="a", header=header, index=False)


# -----------------------------------------------------------------------------
# RUN REAL-DATA EXPERIMENT
# -----------------------------------------------------------------------------

def compute_baselines(graphs, structure_df):
    existing = {}
    if BASELINE_PATH.exists():
        try:
            old = pd.read_csv(BASELINE_PATH)
            existing = {
                str(r["graph"]): (float(r["baseline_hitting_time"]),
                                  float(r["baseline_estimator_se"]))
                for _, r in old.iterrows()
            }
        except Exception:
            existing = {}

    rows = []

    for name, G in graphs.items():
        if name in existing:
            ht, se = existing[name]
            print(f"Baseline {name}: using saved value {ht:.6g}")
        else:
            print(f"Baseline {name}: computing...")
            ht, se = average_hitting_time(
                G,
                probes=TRACE_PROBES,
                seed=stable_seed(name, "baseline", "hitting")
            )
            print(f"  H0={ht:.6g}, estimator SE={se:.4g}")

        q = float(structure_df.loc[structure_df["graph"] == name, "modularity"].iloc[0])

        rows.append({
            "graph": name,
            "baseline_hitting_time": ht,
            "baseline_estimator_se": se,
            "nodes": G.number_of_nodes(),
            "edges": G.number_of_edges(),
            "modularity": q,
            "trace_probes": TRACE_PROBES,
        })

    out = pd.DataFrame(rows)
    out.to_csv(BASELINE_PATH, index=False)
    return out


def run_experiment(graphs, structure_df, baseline_df):
    existing = load_completed_raw()
    done = completed_keys(existing)

    print("\n" + "=" * 80)
    print("REAL-NETWORK DAMAGE EXPERIMENT")
    print("=" * 80)

    for graph_index, (name, G0) in enumerate(graphs.items(), start=1):
        n_trials = TRIALS[name]
        baseline = float(
            baseline_df.loc[baseline_df["graph"] == name, "baseline_hitting_time"].iloc[0]
        )
        modularity = float(
            structure_df.loc[structure_df["graph"] == name, "modularity"].iloc[0]
        )

        print(
            f"\n[{graph_index}/{len(graphs)}] {name}: "
            f"n={G0.number_of_nodes():,}, m={G0.number_of_edges():,}, "
            f"Q={modularity:.4f}, trials={n_trials}"
        )

        for trial in range(n_trials):
            # One nested damage ordering per model and trial.
            pref_order = preferential_edge_order(
                G0, stable_seed(name, trial, "preferential_order")
            )
            rand_order = random_edge_order(
                G0, stable_seed(name, trial, "random_order")
            )

            trial_rows = []

            for damage_type, edge_order in (
                ("preferential", pref_order),
                ("random", rand_order),
            ):
                for frac in DAMAGE_FRACS:
                    key = (name, trial, damage_type, round(float(frac), 4))
                    if key in done:
                        continue

                    damaged = damage_from_order(G0, edge_order, frac)
                    lcc = largest_connected_component(damaged)

                    ht, est_se = average_hitting_time(
                        lcc,
                        probes=TRACE_PROBES,
                        seed=stable_seed(name, trial, damage_type, frac, "hitting")
                    )

                    trial_rows.append({
                        "graph": name,
                        "trial": trial,
                        "damage_type": damage_type,
                        "damage_frac": float(frac),
                        "hitting_time": ht,
                        "trace_estimator_se": est_se,
                        "relative_ht": ht / baseline,
                        "improvement_pct": 100.0 * (1.0 - ht / baseline),
                        "original_nodes": G0.number_of_nodes(),
                        "original_edges": G0.number_of_edges(),
                        "lcc_nodes": lcc.number_of_nodes(),
                        "lcc_edges": lcc.number_of_edges(),
                        "lcc_fraction": lcc.number_of_nodes() / G0.number_of_nodes(),
                        "modularity": modularity,
                        "damage_seed": stable_seed(name, trial, damage_type + "_order"),
                        "hitting_seed": stable_seed(
                            name, trial, damage_type, frac, "hitting"
                        ),
                    })

                    # A real-network size-matched control is paired to the
                    # preferentially damaged LCC at selected fractions.
                    if (
                        damage_type == "preferential"
                        and round(float(frac), 2) in SIZE_CONTROL_FRACS
                    ):
                        size_key = (
                            name, trial, "size_matched", round(float(frac), 4)
                        )

                        if size_key not in done:
                            control = connected_size_matched_subgraph(
                                G0,
                                target_n=lcc.number_of_nodes(),
                                seed=stable_seed(name, trial, frac, "size_control")
                            )

                            htc, sec = average_hitting_time(
                                control,
                                probes=TRACE_PROBES,
                                seed=stable_seed(
                                    name, trial, frac, "size_control_hitting"
                                )
                            )

                            trial_rows.append({
                                "graph": name,
                                "trial": trial,
                                "damage_type": "size_matched",
                                "damage_frac": float(frac),
                                "hitting_time": htc,
                                "trace_estimator_se": sec,
                                "relative_ht": htc / baseline,
                                "improvement_pct": 100.0 * (1.0 - htc / baseline),
                                "original_nodes": G0.number_of_nodes(),
                                "original_edges": G0.number_of_edges(),
                                "lcc_nodes": control.number_of_nodes(),
                                "lcc_edges": control.number_of_edges(),
                                "lcc_fraction": (
                                    control.number_of_nodes() / G0.number_of_nodes()
                                ),
                                "modularity": modularity,
                                "damage_seed": stable_seed(
                                    name, trial, frac, "size_control"
                                ),
                                "hitting_seed": stable_seed(
                                    name, trial, frac, "size_control_hitting"
                                ),
                            })

            append_rows(trial_rows)

            # Refresh completed keys after every trial so a restart is safe.
            for r in trial_rows:
                done.add((
                    r["graph"],
                    int(r["trial"]),
                    r["damage_type"],
                    round(float(r["damage_frac"]), 4)
                ))

            print(
                f"  completed trial {trial + 1:>2}/{n_trials}",
                flush=True
            )

    return pd.read_csv(RAW_PATH)


# -----------------------------------------------------------------------------
# STATISTICS
# -----------------------------------------------------------------------------

def ci95(values):
    x = np.asarray(pd.Series(values).dropna(), dtype=float)

    if len(x) == 0:
        return np.nan, np.nan, np.nan

    mean = float(x.mean())

    if len(x) == 1:
        return mean, np.nan, np.nan

    half = stats.t.ppf(0.975, len(x) - 1) * x.std(ddof=1) / math.sqrt(len(x))
    return mean, mean - half, mean + half


def benjamini_hochberg(p_values):
    p = np.asarray(p_values, dtype=float)
    adjusted = np.full(len(p), np.nan)

    valid = np.isfinite(p)
    pv = p[valid]

    if len(pv) == 0:
        return adjusted

    order = np.argsort(pv)
    sorted_p = pv[order]
    m = len(sorted_p)

    sorted_adj = sorted_p * m / np.arange(1, m + 1)
    sorted_adj = np.minimum.accumulate(sorted_adj[::-1])[::-1]
    sorted_adj = np.clip(sorted_adj, 0.0, 1.0)

    restored = np.empty(m)
    restored[order] = sorted_adj

    adjusted[np.where(valid)[0]] = restored
    return adjusted


def analyze_results(raw, structure_df):
    main = raw[raw["damage_type"].isin(["preferential", "random"])].copy()

    # -------------------------------------------------------------------------
    # Curves + confidence intervals
    # -------------------------------------------------------------------------
    curve_rows = []

    for (name, dtype, frac), g in main.groupby(
        ["graph", "damage_type", "damage_frac"]
    ):
        mean, lo, hi = ci95(g["relative_ht"])

        curve_rows.append({
            "graph": name,
            "damage_type": dtype,
            "damage_frac": frac,
            "n": len(g),
            "mean_relative_ht": mean,
            "ci95_low": lo,
            "ci95_high": hi,
            "mean_improvement_pct": 100.0 * (1.0 - mean),
            "mean_lcc_fraction": g["lcc_fraction"].mean(),
            "sd_relative_ht": g["relative_ht"].std(ddof=1),
        })

    curve = pd.DataFrame(curve_rows)
    curve.to_csv(RESULTS_DIR / "curve_statistics.csv", index=False)

    # -------------------------------------------------------------------------
    # Sustained antifragility per progressive trajectory
    # -------------------------------------------------------------------------
    main["benefit"] = np.maximum(0.0, 1.0 - main["relative_ht"])

    trial_scores = (
        main.groupby(["graph", "damage_type", "trial"], as_index=False)
        .agg(
            sustained_af=("benefit", "mean"),
            mean_relative_ht=("relative_ht", "mean"),
            min_relative_ht=("relative_ht", "min"),
            mean_lcc_fraction=("lcc_fraction", "mean"),
        )
    )
    trial_scores.to_csv(RESULTS_DIR / "trial_sustained_scores.csv", index=False)

    # -------------------------------------------------------------------------
    # Preferential vs random paired test on sustained score
    # -------------------------------------------------------------------------
    comparison_rows = []

    for name in sorted(main["graph"].unique()):
        pref = trial_scores[
            (trial_scores["graph"] == name)
            & (trial_scores["damage_type"] == "preferential")
        ][["trial", "sustained_af"]].rename(
            columns={"sustained_af": "af_pref"}
        )

        rand = trial_scores[
            (trial_scores["graph"] == name)
            & (trial_scores["damage_type"] == "random")
        ][["trial", "sustained_af"]].rename(
            columns={"sustained_af": "af_random"}
        )

        pair = pref.merge(rand, on="trial", how="inner")

        if len(pair) >= 5:
            diff = pair["af_pref"] - pair["af_random"]

            if np.allclose(diff, 0):
                stat_value, p_value = 0.0, 1.0
            else:
                stat_value, p_value = stats.wilcoxon(
                    pair["af_pref"],
                    pair["af_random"],
                    alternative="greater"
                )
        else:
            stat_value, p_value = np.nan, np.nan

        comparison_rows.append({
            "graph": name,
            "paired_trials": len(pair),
            "mean_af_preferential": pair["af_pref"].mean(),
            "mean_af_random": pair["af_random"].mean(),
            "delta_af_pref_minus_random": (
                pair["af_pref"] - pair["af_random"]
            ).mean(),
            "wilcoxon_stat": stat_value,
            "p_one_sided_pref_gt_random": p_value,
        })

    comparison = pd.DataFrame(comparison_rows)
    comparison["p_bh"] = benjamini_hochberg(
        comparison["p_one_sided_pref_gt_random"].to_numpy()
    )
    comparison["significant_fdr_005"] = comparison["p_bh"] < 0.05
    comparison.to_csv(
        RESULTS_DIR / "preferential_vs_random.csv", index=False
    )

    # -------------------------------------------------------------------------
    # Size-matched LCC control
    # -------------------------------------------------------------------------
    pref_for_size = raw[
        (raw["damage_type"] == "preferential")
        & (raw["damage_frac"].round(2).isin(SIZE_CONTROL_FRACS))
    ][
        ["graph", "trial", "damage_frac", "hitting_time",
         "lcc_nodes", "lcc_fraction"]
    ].rename(columns={
        "hitting_time": "pref_hitting_time",
        "lcc_nodes": "pref_lcc_nodes",
        "lcc_fraction": "pref_lcc_fraction",
    })

    size = raw[
        raw["damage_type"] == "size_matched"
    ][
        ["graph", "trial", "damage_frac", "hitting_time",
         "lcc_nodes", "lcc_fraction"]
    ].rename(columns={
        "hitting_time": "size_hitting_time",
        "lcc_nodes": "size_nodes",
        "lcc_fraction": "size_fraction",
    })

    size_pairs = pref_for_size.merge(
        size, on=["graph", "trial", "damage_frac"], how="inner"
    )

    if len(size_pairs):
        size_pairs["pref_over_size_ht"] = (
            size_pairs["pref_hitting_time"] / size_pairs["size_hitting_time"]
        )

    size_pairs.to_csv(
        RESULTS_DIR / "size_matched_pair_level.csv", index=False
    )

    # IMPORTANT: the 10%, 20%, 30%, and 40% size-control observations
    # are nested points from the same progressive damage trajectory.
    # Aggregate those four ratios WITHIN each independent trial first,
    # then perform inference across independent trial-level summaries.
    size_trial = (
        size_pairs
        .groupby(["graph", "trial"], as_index=False)
        .agg(
            pref_over_size_ht=("pref_over_size_ht", "mean"),
            damage_levels=("damage_frac", "nunique"),
        )
    )

    size_trial.to_csv(
        RESULTS_DIR / "size_matched_trial_level.csv", index=False
    )

    size_rows = []

    for name, g in size_trial.groupby("graph"):
        ratio = g["pref_over_size_ht"].to_numpy(dtype=float)
        diff = 1.0 - ratio  # positive means preferential damage is better

        if len(ratio) >= 5 and not np.allclose(ratio - 1.0, 0):
            stat_value, p_value = stats.wilcoxon(
                ratio - 1.0,
                alternative="less"
            )
        elif len(ratio) >= 5:
            stat_value, p_value = 0.0, 1.0
        else:
            stat_value, p_value = np.nan, np.nan

        nonzero = diff[np.isfinite(diff) & (diff != 0)]
        if len(nonzero):
            ranks = stats.rankdata(np.abs(nonzero), method="average")
            w_pos = ranks[nonzero > 0].sum()
            w_neg = ranks[nonzero < 0].sum()
            rank_biserial = float(
                (w_pos - w_neg) / (w_pos + w_neg)
            ) if (w_pos + w_neg) > 0 else 0.0
        else:
            rank_biserial = 0.0

        mean_ratio, lo, hi = ci95(ratio)

        size_rows.append({
            "graph": name,
            "independent_trials": len(g),
            "mean_pref_over_size_ht": mean_ratio,
            "ci95_low": lo,
            "ci95_high": hi,
            "wilcoxon_stat": stat_value,
            "p_one_sided_ratio_lt_1": p_value,
            "rank_biserial_benefit": rank_biserial,
        })

    size_summary = pd.DataFrame(size_rows)

    if len(size_summary):
        size_summary["p_bh"] = benjamini_hochberg(
            size_summary["p_one_sided_ratio_lt_1"].to_numpy()
        )
        size_summary["significant_fdr_005"] = size_summary["p_bh"] < 0.05

    size_summary.to_csv(
        RESULTS_DIR / "size_matched_controls.csv", index=False
    )

    # -------------------------------------------------------------------------
    # Network-level summary
    # -------------------------------------------------------------------------
    summary_rows = []

    for name in sorted(main["graph"].unique()):
        q = float(
            structure_df.loc[
                structure_df["graph"] == name, "modularity"
            ].iloc[0]
        )

        pc = curve[
            (curve["graph"] == name)
            & (curve["damage_type"] == "preferential")
        ].sort_values("damage_frac")

        if pc.empty:
            continue

        best_idx = pc["mean_relative_ht"].idxmin()
        best_rel = float(pc.loc[best_idx, "mean_relative_ht"])

        if best_rel < 1.0:
            max_improvement = 100.0 * (1.0 - best_rel)
            best_damage = 100.0 * float(pc.loc[best_idx, "damage_frac"])
        else:
            max_improvement = 0.0
            best_damage = np.nan

        pref_scores = trial_scores[
            (trial_scores["graph"] == name)
            & (trial_scores["damage_type"] == "preferential")
        ]["sustained_af"]

        rand_scores = trial_scores[
            (trial_scores["graph"] == name)
            & (trial_scores["damage_type"] == "random")
        ]["sustained_af"]

        comp = comparison[comparison["graph"] == name]
        sc = size_summary[size_summary["graph"] == name]

        summary_rows.append({
            "graph": name,
            "modularity": q,
            "trials": TRIALS[name],
            "max_mean_improvement_pct": max_improvement,
            "best_damage_pct": best_damage,
            "damage_levels_ge_2pct": int(
                (pc["mean_relative_ht"] < IMPROVEMENT_THRESHOLD).sum()
            ),
            "total_damage_levels": len(DAMAGE_FRACS),
            "mean_sustained_af_preferential": pref_scores.mean(),
            "mean_sustained_af_random": rand_scores.mean(),
            "pref_vs_random_p_bh": (
                float(comp["p_bh"].iloc[0]) if len(comp) else np.nan
            ),
            "mean_lcc_fraction_at_best": float(
                pc.loc[best_idx, "mean_lcc_fraction"]
            ),
            "mean_pref_over_size_ht": (
                float(sc["mean_pref_over_size_ht"].iloc[0])
                if len(sc) else np.nan
            ),
            "size_control_p_bh": (
                float(sc["p_bh"].iloc[0])
                if len(sc) and "p_bh" in sc.columns else np.nan
            ),
        })

    network_summary = pd.DataFrame(summary_rows)
    network_summary.to_csv(
        RESULTS_DIR / "network_summary.csv", index=False
    )

    # -------------------------------------------------------------------------
    # Cross-network modularity association
    # -------------------------------------------------------------------------
    valid = network_summary[
        ["modularity", "mean_sustained_af_preferential"]
    ].dropna()

    if len(valid) >= 3:
        rho, p_rho = stats.spearmanr(
            valid["modularity"],
            valid["mean_sustained_af_preferential"]
        )
    else:
        rho, p_rho = np.nan, np.nan

    mod_assoc = pd.DataFrame([{
        "n_real_networks": len(valid),
        "spearman_rho_modularity_vs_sustained_af": rho,
        "p_value": p_rho,
    }])

    mod_assoc.to_csv(
        RESULTS_DIR / "modularity_association.csv", index=False
    )

    return (
        curve,
        trial_scores,
        comparison,
        size_pairs,
        size_summary,
        network_summary,
        rho,
        p_rho,
    )


# -----------------------------------------------------------------------------
# FIGURES
# -----------------------------------------------------------------------------

def save_current_figure(stem):
    plt.tight_layout()
    plt.savefig(FIGS_DIR / f"{stem}.png", dpi=300, bbox_inches="tight")
    plt.savefig(FIGS_DIR / f"{stem}.pdf", bbox_inches="tight")
    plt.close()


def make_figures(curve, structure_df, network_summary, size_pairs):
    # One publication figure per real network: preferential vs random.
    for name in structure_df["graph"]:
        g = curve[curve["graph"] == name]

        if g.empty:
            continue

        plt.figure(figsize=(7.2, 4.8))

        for dtype in ("preferential", "random"):
            sub = g[g["damage_type"] == dtype].sort_values("damage_frac")

            x = sub["damage_frac"].to_numpy() * 100.0
            y = sub["mean_relative_ht"].to_numpy()
            lo = sub["ci95_low"].to_numpy()
            hi = sub["ci95_high"].to_numpy()

            plt.plot(
                x, y, marker="o", linewidth=1.8,
                label=dtype.capitalize()
            )
            plt.fill_between(x, lo, hi, alpha=0.15)

        plt.axhline(1.0, linestyle="--", linewidth=1.0)
        q = float(
            structure_df.loc[
                structure_df["graph"] == name, "modularity"
            ].iloc[0]
        )

        plt.xlabel("Edge Damage Fraction (%)")
        plt.ylabel("Relative Average Hitting Time")
        plt.title(f"{name}: Preferential vs Random Damage (Q={q:.3f})")
        plt.legend()
        plt.grid(True, alpha=0.25)

        save_current_figure(
            f"{name.lower()}_preferential_vs_random"
        )

    # LCC retention across all real networks.
    pref = curve[curve["damage_type"] == "preferential"]

    plt.figure(figsize=(7.6, 5.0))

    for name in sorted(pref["graph"].unique()):
        sub = pref[pref["graph"] == name].sort_values("damage_frac")
        plt.plot(
            sub["damage_frac"] * 100.0,
            sub["mean_lcc_fraction"],
            marker="o",
            linewidth=1.5,
            label=name
        )

    plt.xlabel("Edge Damage Fraction (%)")
    plt.ylabel("Fraction of Original Nodes in LCC")
    plt.ylim(0.0, 1.03)
    plt.title("Largest Connected Component Retention")
    plt.legend(fontsize=7)
    plt.grid(True, alpha=0.25)
    save_current_figure("lcc_retention_all_real_networks")

    # Modularity vs sustained response.
    plt.figure(figsize=(7.2, 4.8))
    plt.scatter(
        network_summary["modularity"],
        network_summary["mean_sustained_af_preferential"],
        s=60
    )

    for _, r in network_summary.iterrows():
        plt.annotate(
            r["graph"],
            (
                r["modularity"],
                r["mean_sustained_af_preferential"]
            ),
            xytext=(4, 4),
            textcoords="offset points",
            fontsize=8
        )

    plt.xlabel("Louvain Modularity Q")
    plt.ylabel("Mean Sustained Antifragility Score")
    plt.title("Real Networks: Modularity and Sustained Response")
    plt.grid(True, alpha=0.25)
    save_current_figure("modularity_vs_sustained_af_real_only")

    # Size-matched control distribution.
    if len(size_pairs):
        plt.figure(figsize=(7.6, 5.0))

        groups = []
        labels = []

        for name in sorted(size_pairs["graph"].unique()):
            vals = size_pairs.loc[
                size_pairs["graph"] == name,
                "pref_over_size_ht"
            ].dropna().to_numpy()

            if len(vals):
                groups.append(vals)
                labels.append(name)

        if groups:
            plt.boxplot(groups, labels=labels, showfliers=False)
            plt.axhline(1.0, linestyle="--", linewidth=1.0)
            plt.ylabel(
                "Preferential-Damage Hitting Time / "
                "Size-Matched Control Hitting Time"
            )
            plt.xticks(rotation=25, ha="right")
            plt.title("LCC Size-Matched Control")
            plt.grid(True, axis="y", alpha=0.25)
            save_current_figure("size_matched_control_real_networks")
        else:
            plt.close()


# -----------------------------------------------------------------------------
# AUTOMATIC LATEX RESULTS
# -----------------------------------------------------------------------------

def latex_escape(text):
    return str(text).replace("_", r"\_")


def latex_number(value, digits=3):
    if pd.isna(value):
        return "--"
    return f"{float(value):.{digits}f}"


def latex_p(value):
    if pd.isna(value):
        return "--"
    value = float(value)
    if value < 0.001:
        return r"$<0.001$"
    return f"{value:.3f}"


def write_results_auto(
    provenance_df,
    structure_df,
    network_summary,
    comparison,
    size_summary,
    rho,
    p_rho,
):
    out = RESULTS_DIR / "results_auto.tex"
    lines = []

    lines.append("% AUTO-GENERATED FROM REAL DATA ONLY. DO NOT EDIT VALUES BY HAND.")
    lines.append(
        rf"\newcommand{{\RealNetworkCount}}{{{len(network_summary)}}}"
    )
    lines.append(
        rf"\newcommand{{\RealSpearmanRho}}{{{latex_number(rho, 3)}}}"
    )
    lines.append(
        rf"\newcommand{{\RealSpearmanP}}{{{latex_p(p_rho)}}}"
    )
    lines.append("")

    merged = (
        structure_df
        .merge(network_summary, on=["graph", "modularity"], how="inner")
        .sort_values("modularity", ascending=False)
    )

    lines.extend([
        r"\begin{table*}[!t]",
        r"\centering",
        r"\caption{Real-network structure and preferential-damage response.}",
        r"\label{tab:real_summary}",
        r"\begin{tabular}{lrrrrrrrr}",
        r"\toprule",
        r"Network & $n$ & $m$ & $Q$ & Max Imp. & $\geq$2\% Levels & "
        r"$A_{\rm pref}$ & $A_{\rm rand}$ & BH-$p$ \\",
        r"\midrule",
    ])

    for _, r in merged.iterrows():
        lines.append(
            f"{latex_escape(r['graph'])} & "
            f"{int(r['nodes'])} & "
            f"{int(r['edges'])} & "
            f"{r['modularity']:.3f} & "
            f"{r['max_mean_improvement_pct']:.1f}\\% & "
            f"{int(r['damage_levels_ge_2pct'])}/"
            f"{int(r['total_damage_levels'])} & "
            f"{r['mean_sustained_af_preferential']:.3f} & "
            f"{r['mean_sustained_af_random']:.3f} & "
            f"{latex_p(r['pref_vs_random_p_bh'])} \\\\"
        )

    lines.extend([
        r"\bottomrule",
        r"\end{tabular}",
        r"\end{table*}",
        "",
        r"\begin{table}[!t]",
        r"\centering",
        r"\caption{Largest-component size-matched control.}",
        r"\label{tab:size_matched}",
        r"\begin{tabular}{lrr}",
        r"\toprule",
        r"Network & $T_{\rm pref}/T_{\rm size}$ & BH-$p$ \\",
        r"\midrule",
    ])

    for _, r in size_summary.sort_values("graph").iterrows():
        lines.append(
            f"{latex_escape(r['graph'])} & "
            f"{latex_number(r['mean_pref_over_size_ht'], 3)} & "
            f"{latex_p(r['p_bh'])} \\\\"
        )

    lines.extend([
        r"\bottomrule",
        r"\end{tabular}",
        r"\end{table}",
        "",
    ])

    out.write_text("\n".join(lines), encoding="utf-8")
    print(f"Saved automatic LaTeX results: {out}")


# -----------------------------------------------------------------------------
# FINAL AUDIT
# -----------------------------------------------------------------------------

def audit_outputs(
    graphs,
    provenance_df,
    structure_df,
    baseline_df,
    raw,
    network_summary,
):
    print("\n" + "=" * 80)
    print("FINAL DATA-INTEGRITY AUDIT")
    print("=" * 80)

    expected_graphs = set(DATASETS.keys())

    checks = {
        "all configured real datasets downloaded":
            set(provenance_df["graph"]) == expected_graphs,

        "all network structure rows present":
            set(structure_df["graph"]) == expected_graphs,

        "all baselines present":
            set(baseline_df["graph"]) == expected_graphs,

        "all summary rows present":
            set(network_summary["graph"]) == expected_graphs,

        "raw results contain no unknown graph names":
            set(raw["graph"]).issubset(expected_graphs),

        "damage types restricted to empirical transformations":
            set(raw["damage_type"]).issubset(
                {"preferential", "random", "size_matched"}
            ),

        "no f=0 rows mixed into damaged raw data":
            bool((raw["damage_frac"] > 0).all()),

        "relative hitting times finite":
            bool(np.isfinite(raw["relative_ht"]).all()),

        "LCC fractions within (0,1]":
            bool(((raw["lcc_fraction"] > 0) &
                  (raw["lcc_fraction"] <= 1)).all()),
    }

    failed = []

    for label, ok in checks.items():
        print(f"  {'PASS' if ok else 'FAIL'} — {label}")
        if not ok:
            failed.append(label)

    print("\nSynthetic graph generators used: NONE")
    print("Hardcoded experimental outcomes: NONE")
    print("Source data provenance + SHA-256 hashes:",
          RESULTS_DIR / "dataset_provenance.csv")

    if failed:
        raise RuntimeError(
            "Final audit failed: " + "; ".join(failed)
        )

    print("\nALL FINAL AUDIT CHECKS PASSED.")


# -----------------------------------------------------------------------------
# MAIN
# -----------------------------------------------------------------------------

print("=" * 80)
print("MARKOVIAN ANTIFRAGILITY — REAL-DATA-ONLY IEEE EMPIRICAL STUDY")
print("=" * 80)
print("Synthetic networks: NONE")
print("Hardcoded empirical results: NONE")
print("Source repository: Stanford SNAP")
print(f"Output directory: {RESULTS_DIR}")

# 1. Download real source data and construct analysis graphs.
graphs, provenance_df = load_all_real_networks()

# 2. Compute real-network structural descriptors.
print("\n" + "=" * 80)
print("NETWORK STRUCTURAL CHARACTERIZATION")
print("=" * 80)

structure_rows = []

for name, G in graphs.items():
    print(f"Computing structure: {name}")
    row = network_descriptors(G, name)
    structure_rows.append(row)
    print(
        f"  n={row['nodes']:,}, m={row['edges']:,}, "
        f"Q={row['modularity']:.4f}, communities={row['communities']}"
    )

structure_df = pd.DataFrame(structure_rows)
structure_df.to_csv(
    RESULTS_DIR / "network_structure.csv", index=False
)

# 3. Compute/save undamaged baselines.
baseline_df = compute_baselines(graphs, structure_df)

# 4. Generate the realization-level empirical data.
raw = run_experiment(graphs, structure_df, baseline_df)

# 5. Statistical analysis.
(
    curve,
    trial_scores,
    comparison,
    size_pairs,
    size_summary,
    network_summary,
    rho,
    p_rho,
) = analyze_results(raw, structure_df)

# 6. Publication figures.
make_figures(
    curve,
    structure_df,
    network_summary,
    size_pairs
)

# 7. Automatic LaTeX tables/macros.
write_results_auto(
    provenance_df,
    structure_df,
    network_summary,
    comparison,
    size_summary,
    rho,
    p_rho,
)

# 8. Final validation.
audit_outputs(
    graphs,
    provenance_df,
    structure_df,
    baseline_df,
    raw,
    network_summary,
)

print("\n" + "=" * 80)
print("FINAL EMPIRICAL STUDY COMPLETE")
print("=" * 80)

print("\nNetwork summary:")
print(network_summary.to_string(index=False))

print(
    f"\nSpearman modularity vs sustained preferential AF: "
    f"rho={rho:.4f}, p={p_rho:.6g}"
)

print("\nPrimary outputs:")
for filename in [
    "dataset_provenance.csv",
    "network_structure.csv",
    "baseline_transport.csv",
    "real_networks_raw.csv",
    "curve_statistics.csv",
    "trial_sustained_scores.csv",
    "preferential_vs_random.csv",
    "size_matched_pair_level.csv",
    "size_matched_trial_level.csv",
    "size_matched_controls.csv",
    "network_summary.csv",
    "modularity_association.csv",
    "results_auto.tex",
]:
    print(" ", RESULTS_DIR / filename)

print("\nFigures:")
print(" ", FIGS_DIR)

print("\nNo synthetic datasets or synthetic network generators were used.")

Mounted at /content/drive
MARKOVIAN ANTIFRAGILITY — REAL-DATA-ONLY IEEE EMPIRICAL STUDY
Synthetic networks: NONE
Hardcoded empirical results: NONE
Source repository: Stanford SNAP
Output directory: /content/drive/MyDrive/Markov_Antifragility_IEEE/results_real_only

DOWNLOADING / LOADING REAL PUBLIC DATASETS

ca_GrQc
  existing: ca-GrQc.txt.gz
  raw observed rows: 28,980
  simple graph:      n=5,241, m=14,484
  analysis LCC:      n=4,158, m=13,422

ca_HepTh
  existing: ca-HepTh.txt.gz
  raw observed rows: 51,971
  simple graph:      n=9,875, m=25,973
  analysis LCC:      n=8,638, m=24,806

Facebook_Combined
  existing: facebook_combined.txt.gz
  raw observed rows: 88,234
  simple graph:      n=4,039, m=88,234
  analysis LCC:      n=4,039, m=88,234

Email_Eu_Core
  existing: email-Eu-core.txt.gz
  raw observed rows: 25,571
  simple graph:      n=986, m=16,064
  analysis LCC:      n=986, m=16,064

Wiki_Vote
  existing: wiki-Vote.txt.gz
  raw observed rows: 103,689
  simple graph:      n=7

/tmp/ipykernel_3852/2188823699.py:1189: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(groups, labels=labels, showfliers=False)


Saved automatic LaTeX results: /content/drive/MyDrive/Markov_Antifragility_IEEE/results_real_only/results_auto.tex

FINAL DATA-INTEGRITY AUDIT
  PASS — all configured real datasets downloaded
  PASS — all network structure rows present
  PASS — all baselines present
  PASS — all summary rows present
  PASS — raw results contain no unknown graph names
  PASS — damage types restricted to empirical transformations
  PASS — no f=0 rows mixed into damaged raw data
  PASS — relative hitting times finite
  PASS — LCC fractions within (0,1]

Synthetic graph generators used: NONE
Hardcoded experimental outcomes: NONE
Source data provenance + SHA-256 hashes: /content/drive/MyDrive/Markov_Antifragility_IEEE/results_real_only/dataset_provenance.csv

ALL FINAL AUDIT CHECKS PASSED.

FINAL EMPIRICAL STUDY COMPLETE

Network summary:
            graph  modularity  trials  max_mean_improvement_pct  best_damage_pct  damage_levels_ge_2pct  total_damage_levels  mean_sustained_af_preferential  mean_sustaine

In [2]:
import os
import math
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

from scipy import sparse
from scipy.sparse.csgraph import shortest_path
from scipy.stats import wilcoxon, rankdata

# -----------------------------------------------------------------------------
# CONFIGURATION
# -----------------------------------------------------------------------------

CONFIRM_DIR = RESULTS_DIR / "confirmatory_benchmarks"
CONFIRM_FIG_DIR = CONFIRM_DIR / "figures"
CONFIRM_DIR.mkdir(parents=True, exist_ok=True)
CONFIRM_FIG_DIR.mkdir(parents=True, exist_ok=True)

# Use the same representative fractions already used for size-matched controls.
CONFIRM_FRACS = [0.10, 0.20, 0.30, 0.40]

# Number of uniformly sampled source nodes used to estimate global efficiency.
# 24 keeps the benchmark computationally practical while remaining fully
# deterministic/reproducible. Increase to 48 or 64 for a sensitivity analysis.
EFFICIENCY_SOURCES = 24

ALPHA = 0.05
MIN_LCC_FRACTION = 0.80
MIN_IMPROVEMENT_LEVELS = 3

print("=" * 80)
print("CONFIRMATORY BENCHMARK ANALYSIS")
print("=" * 80)
print("Synthetic networks: NONE")
print("Hardcoded empirical outcomes: NONE")
print(f"Global-efficiency source samples per graph state: {EFFICIENCY_SOURCES}")
print(f"Confirmatory damage fractions: {CONFIRM_FRACS}")

# -----------------------------------------------------------------------------
# LOAD SAVED PRIMARY RESULTS
# -----------------------------------------------------------------------------

RAW_PATH = RESULTS_DIR / "real_networks_raw.csv"
SUMMARY_PATH = RESULTS_DIR / "network_summary.csv"
PVR_PATH = RESULTS_DIR / "preferential_vs_random.csv"
SIZE_PATH = RESULTS_DIR / "size_matched_pair_level.csv"

for required in (RAW_PATH, SUMMARY_PATH, PVR_PATH, SIZE_PATH):
    if not required.exists():
        raise FileNotFoundError(f"Required primary output is missing: {required}")

raw_primary = pd.read_csv(RAW_PATH)
network_summary = pd.read_csv(SUMMARY_PATH)
pref_random_primary = pd.read_csv(PVR_PATH)
size_pair_primary = pd.read_csv(SIZE_PATH)

# If this cell is run after a notebook restart, rebuild the same real graphs
# using the main pipeline's public-data loader.
if "graphs" not in globals():
    print("Graph objects not in memory; reloading the same real SNAP datasets...")
    graphs, _confirm_provenance = load_all_real_networks()

# -----------------------------------------------------------------------------
# STATISTICAL HELPERS
# -----------------------------------------------------------------------------

def bh_adjust(pvalues):
    """Benjamini-Hochberg FDR-adjusted p-values."""
    p = np.asarray(pvalues, dtype=float)
    m = len(p)
    if m == 0:
        return np.array([], dtype=float)

    order = np.argsort(p)
    ranked = p[order]
    adjusted = ranked * m / np.arange(1, m + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    adjusted = np.clip(adjusted, 0.0, 1.0)

    out = np.empty(m, dtype=float)
    out[order] = adjusted
    return out


def matched_rank_biserial(x, y):
    """
    Matched-pairs rank-biserial correlation.
    Positive values mean x > y.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    d = x - y
    d = d[np.isfinite(d) & (d != 0)]

    if len(d) == 0:
        return 0.0

    ranks = rankdata(np.abs(d), method="average")
    w_pos = ranks[d > 0].sum()
    w_neg = ranks[d < 0].sum()
    denom = w_pos + w_neg
    return float((w_pos - w_neg) / denom) if denom > 0 else 0.0


def safe_wilcoxon_greater(x, y):
    """One-sided paired Wilcoxon test for x > y."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    x, y = x[mask], y[mask]

    if len(x) == 0:
        return np.nan, np.nan

    d = x - y
    if np.allclose(d, 0):
        return 0.0, 1.0

    res = wilcoxon(
        x, y,
        alternative="greater",
        zero_method="wilcox",
        correction=False,
        method="auto",
    )
    return float(res.statistic), float(res.pvalue)


def t_ci(values, alpha=0.05):
    """Two-sided t confidence interval for an across-trial mean."""
    from scipy.stats import t

    x = np.asarray(values, dtype=float)
    x = x[np.isfinite(x)]

    if len(x) == 0:
        return np.nan, np.nan, np.nan

    mean = float(np.mean(x))
    if len(x) == 1:
        return mean, np.nan, np.nan

    se = float(np.std(x, ddof=1) / np.sqrt(len(x)))
    crit = float(t.ppf(1.0 - alpha / 2.0, df=len(x) - 1))
    return mean, mean - crit * se, mean + crit * se


# -----------------------------------------------------------------------------
# GLOBAL EFFICIENCY ESTIMATOR
# -----------------------------------------------------------------------------

def sampled_global_efficiency(G, n_sources=24, seed=0):
    """
    Deterministic source-sampled estimate of global efficiency.

    For uniformly sampled source nodes s:
        e_s = [1/(n-1)] * sum_{t != s} 1/d(s,t)

    The estimator is mean_s(e_s).

    G is expected to be connected (the experiment uses the LCC).
    """
    n = G.number_of_nodes()
    if n <= 1:
        return np.nan, np.nan, 0

    nodes = list(G.nodes())
    rng = np.random.default_rng(seed)

    k = min(int(n_sources), n)
    src_pos = np.sort(rng.choice(n, size=k, replace=False))

    A = nx.to_scipy_sparse_array(
        G,
        nodelist=nodes,
        dtype=np.float64,
        format="csr",
        weight=None,
    )

    # Sparse unweighted shortest paths from sampled sources to all nodes.
    D = shortest_path(
        csgraph=A,
        directed=False,
        unweighted=True,
        indices=src_pos,
        return_predecessors=False,
    )

    efficiencies = np.empty(k, dtype=float)

    for i in range(k):
        d = np.asarray(D[i], dtype=float)
        valid = np.isfinite(d) & (d > 0)

        # Because the graph is connected, there should be n-1 valid targets.
        inv = np.zeros_like(d, dtype=float)
        inv[valid] = 1.0 / d[valid]
        efficiencies[i] = inv.sum() / (n - 1.0)

    est = float(np.mean(efficiencies))
    source_se = (
        float(np.std(efficiencies, ddof=1) / np.sqrt(k))
        if k > 1 else np.nan
    )
    return est, source_se, k


# -----------------------------------------------------------------------------
# RECONSTRUCT THE SAME EMPIRICAL DAMAGE STATES
# -----------------------------------------------------------------------------

EFF_RAW_PATH = CONFIRM_DIR / "global_efficiency_raw.csv"

if EFF_RAW_PATH.exists():
    eff_raw = pd.read_csv(EFF_RAW_PATH)
    print(f"Resuming existing confirmatory results: {EFF_RAW_PATH}")
else:
    eff_raw = pd.DataFrame()

done_eff = set()
if not eff_raw.empty:
    done_eff = set(
        zip(
            eff_raw["graph"],
            eff_raw["trial"].astype(int),
            eff_raw["damage_type"],
            eff_raw["damage_frac"].round(4),
        )
    )

# Baseline efficiencies.
baseline_eff_rows = []

for name, G0 in graphs.items():
    e0, e0_se, k0 = sampled_global_efficiency(
        G0,
        n_sources=EFFICIENCY_SOURCES,
        seed=stable_seed(name, "baseline_global_efficiency"),
    )
    baseline_eff_rows.append({
        "graph": name,
        "baseline_global_efficiency": e0,
        "baseline_source_sampling_se": e0_se,
        "efficiency_sources": k0,
    })

baseline_eff = pd.DataFrame(baseline_eff_rows)
baseline_eff.to_csv(
    CONFIRM_DIR / "baseline_global_efficiency.csv",
    index=False
)

new_rows = []

for graph_index, (name, G0) in enumerate(graphs.items(), start=1):
    n_trials = int(TRIALS[name])
    e0 = float(
        baseline_eff.loc[
            baseline_eff["graph"] == name,
            "baseline_global_efficiency"
        ].iloc[0]
    )

    print(
        f"\n[{graph_index}/{len(graphs)}] {name}: "
        f"confirmatory global efficiency, trials={n_trials}"
    )

    for trial in range(n_trials):

        pref_order = preferential_edge_order(
            G0,
            stable_seed(name, trial, "preferential_order")
        )
        rand_order = random_edge_order(
            G0,
            stable_seed(name, trial, "random_order")
        )

        for frac in CONFIRM_FRACS:

            # -------------------------------------------------------------
            # Preferential damaged LCC
            # -------------------------------------------------------------
            pref_key = (name, trial, "preferential", round(frac, 4))

            pref_lcc = None
            if pref_key not in done_eff:
                pref_damaged = damage_from_order(G0, pref_order, frac)
                pref_lcc = largest_connected_component(pref_damaged)

                e_pref, se_pref, k_pref = sampled_global_efficiency(
                    pref_lcc,
                    n_sources=EFFICIENCY_SOURCES,
                    seed=stable_seed(
                        name, trial, "preferential", frac,
                        "global_efficiency"
                    ),
                )

                new_rows.append({
                    "graph": name,
                    "trial": trial,
                    "damage_type": "preferential",
                    "damage_frac": frac,
                    "global_efficiency": e_pref,
                    "source_sampling_se": se_pref,
                    "efficiency_sources": k_pref,
                    "baseline_global_efficiency": e0,
                    "relative_global_efficiency": e_pref / e0,
                    "efficiency_improvement_pct": 100.0 * (e_pref / e0 - 1.0),
                    "lcc_nodes": pref_lcc.number_of_nodes(),
                    "lcc_fraction": pref_lcc.number_of_nodes() / G0.number_of_nodes(),
                })

            # -------------------------------------------------------------
            # Random damaged LCC
            # -------------------------------------------------------------
            rand_key = (name, trial, "random", round(frac, 4))

            if rand_key not in done_eff:
                rand_damaged = damage_from_order(G0, rand_order, frac)
                rand_lcc = largest_connected_component(rand_damaged)

                e_rand, se_rand, k_rand = sampled_global_efficiency(
                    rand_lcc,
                    n_sources=EFFICIENCY_SOURCES,
                    seed=stable_seed(
                        name, trial, "random", frac,
                        "global_efficiency"
                    ),
                )

                new_rows.append({
                    "graph": name,
                    "trial": trial,
                    "damage_type": "random",
                    "damage_frac": frac,
                    "global_efficiency": e_rand,
                    "source_sampling_se": se_rand,
                    "efficiency_sources": k_rand,
                    "baseline_global_efficiency": e0,
                    "relative_global_efficiency": e_rand / e0,
                    "efficiency_improvement_pct": 100.0 * (e_rand / e0 - 1.0),
                    "lcc_nodes": rand_lcc.number_of_nodes(),
                    "lcc_fraction": rand_lcc.number_of_nodes() / G0.number_of_nodes(),
                })

            # -------------------------------------------------------------
            # Same real-network size-matched control
            # -------------------------------------------------------------
            size_key = (name, trial, "size_matched", round(frac, 4))

            if size_key not in done_eff:
                if pref_lcc is None:
                    pref_damaged = damage_from_order(G0, pref_order, frac)
                    pref_lcc = largest_connected_component(pref_damaged)

                control = connected_size_matched_subgraph(
                    G0,
                    target_n=pref_lcc.number_of_nodes(),
                    seed=stable_seed(name, trial, frac, "size_control")
                )

                e_size, se_size, k_size = sampled_global_efficiency(
                    control,
                    n_sources=EFFICIENCY_SOURCES,
                    seed=stable_seed(
                        name, trial, frac,
                        "size_control_global_efficiency"
                    ),
                )

                new_rows.append({
                    "graph": name,
                    "trial": trial,
                    "damage_type": "size_matched",
                    "damage_frac": frac,
                    "global_efficiency": e_size,
                    "source_sampling_se": se_size,
                    "efficiency_sources": k_size,
                    "baseline_global_efficiency": e0,
                    "relative_global_efficiency": e_size / e0,
                    "efficiency_improvement_pct": 100.0 * (e_size / e0 - 1.0),
                    "lcc_nodes": control.number_of_nodes(),
                    "lcc_fraction": control.number_of_nodes() / G0.number_of_nodes(),
                })

        # Incremental checkpoint after every trial.
        if new_rows:
            add = pd.DataFrame(new_rows)
            eff_raw = pd.concat([eff_raw, add], ignore_index=True)
            eff_raw = eff_raw.drop_duplicates(
                subset=["graph", "trial", "damage_type", "damage_frac"],
                keep="last"
            )
            eff_raw.to_csv(EFF_RAW_PATH, index=False)

            for r in new_rows:
                done_eff.add(
                    (
                        r["graph"],
                        int(r["trial"]),
                        r["damage_type"],
                        round(float(r["damage_frac"]), 4),
                    )
                )
            new_rows = []

        print(f"  completed trial {trial + 1:2d}/{n_trials}")

# -----------------------------------------------------------------------------
# CURVE-LEVEL GLOBAL EFFICIENCY SUMMARY
# -----------------------------------------------------------------------------

curve_rows = []

for (graph, dtype, frac), g in eff_raw.groupby(
    ["graph", "damage_type", "damage_frac"]
):
    mean, lo, hi = t_ci(g["relative_global_efficiency"])
    curve_rows.append({
        "graph": graph,
        "damage_type": dtype,
        "damage_frac": frac,
        "trials": len(g),
        "mean_relative_global_efficiency": mean,
        "ci95_low": lo,
        "ci95_high": hi,
        "mean_efficiency_improvement_pct": 100.0 * (mean - 1.0),
        "mean_lcc_fraction": float(g["lcc_fraction"].mean()),
    })

eff_curve = pd.DataFrame(curve_rows)
eff_curve.to_csv(
    CONFIRM_DIR / "global_efficiency_curve_statistics.csv",
    index=False
)

# -----------------------------------------------------------------------------
# PER-TRIAL SUSTAINED GLOBAL-EFFICIENCY GAIN
#
# Higher global efficiency is better, so:
#   A_E = mean_f max(0, E_rel(f) - 1)
# -----------------------------------------------------------------------------

main_eff = eff_raw[
    eff_raw["damage_type"].isin(["preferential", "random"])
].copy()

trial_eff_rows = []

for (graph, dtype, trial), g in main_eff.groupby(
    ["graph", "damage_type", "trial"]
):
    vals = g["relative_global_efficiency"].to_numpy(dtype=float)
    trial_eff_rows.append({
        "graph": graph,
        "damage_type": dtype,
        "trial": int(trial),
        "sustained_efficiency_gain": float(np.mean(np.maximum(0.0, vals - 1.0))),
        "mean_relative_global_efficiency": float(np.mean(vals)),
        "max_relative_global_efficiency": float(np.max(vals)),
        "mean_lcc_fraction": float(g["lcc_fraction"].mean()),
    })

trial_eff = pd.DataFrame(trial_eff_rows)
trial_eff.to_csv(
    CONFIRM_DIR / "trial_sustained_global_efficiency.csv",
    index=False
)

# -----------------------------------------------------------------------------
# PREFERENTIAL VS RANDOM GLOBAL EFFICIENCY
# -----------------------------------------------------------------------------

pvr_eff_rows = []

for graph in sorted(main_eff["graph"].unique()):

    p = trial_eff[
        (trial_eff["graph"] == graph) &
        (trial_eff["damage_type"] == "preferential")
    ][["trial", "sustained_efficiency_gain"]].rename(
        columns={"sustained_efficiency_gain": "pref"}
    )

    r = trial_eff[
        (trial_eff["graph"] == graph) &
        (trial_eff["damage_type"] == "random")
    ][["trial", "sustained_efficiency_gain"]].rename(
        columns={"sustained_efficiency_gain": "random"}
    )

    pair = p.merge(r, on="trial", how="inner")

    stat, pval = safe_wilcoxon_greater(pair["pref"], pair["random"])
    rb = matched_rank_biserial(pair["pref"], pair["random"])

    delta = pair["pref"] - pair["random"]
    dmean, dlo, dhi = t_ci(delta)

    pvr_eff_rows.append({
        "graph": graph,
        "paired_trials": len(pair),
        "mean_sustained_efficiency_preferential": float(pair["pref"].mean()),
        "mean_sustained_efficiency_random": float(pair["random"].mean()),
        "delta_pref_minus_random": dmean,
        "delta_ci95_low": dlo,
        "delta_ci95_high": dhi,
        "wilcoxon_stat": stat,
        "p_one_sided_pref_gt_random": pval,
        "rank_biserial_effect": rb,
    })

pvr_eff = pd.DataFrame(pvr_eff_rows)
pvr_eff["p_bh"] = bh_adjust(
    pvr_eff["p_one_sided_pref_gt_random"].to_numpy()
)
pvr_eff["significant_fdr_005"] = pvr_eff["p_bh"] < ALPHA

pvr_eff.to_csv(
    CONFIRM_DIR / "global_efficiency_preferential_vs_random.csv",
    index=False
)

# -----------------------------------------------------------------------------
# PREFERENTIAL VS SIZE-MATCHED GLOBAL EFFICIENCY
# -----------------------------------------------------------------------------

pref_eff = eff_raw[
    eff_raw["damage_type"] == "preferential"
][
    ["graph", "trial", "damage_frac", "global_efficiency"]
].rename(columns={"global_efficiency": "pref_efficiency"})

size_eff = eff_raw[
    eff_raw["damage_type"] == "size_matched"
][
    ["graph", "trial", "damage_frac", "global_efficiency"]
].rename(columns={"global_efficiency": "size_efficiency"})

size_eff_pairs = pref_eff.merge(
    size_eff,
    on=["graph", "trial", "damage_frac"],
    how="inner"
)
size_eff_pairs["pref_over_size_efficiency"] = (
    size_eff_pairs["pref_efficiency"] /
    size_eff_pairs["size_efficiency"]
)

size_eff_pairs.to_csv(
    CONFIRM_DIR / "global_efficiency_size_matched_pair_level.csv",
    index=False
)

# The four size-control fractions are nested within each progressive trial.
# Aggregate within trial before any inferential test.
size_eff_trial = (
    size_eff_pairs
    .groupby(["graph", "trial"], as_index=False)
    .agg(
        pref_over_size_efficiency=(
            "pref_over_size_efficiency", "mean"
        ),
        damage_levels=("damage_frac", "nunique"),
    )
)

size_eff_trial.to_csv(
    CONFIRM_DIR / "global_efficiency_size_matched_trial_level.csv",
    index=False
)

size_eff_rows = []

for graph, g in size_eff_trial.groupby("graph"):
    ratio = g["pref_over_size_efficiency"].to_numpy(dtype=float)

    # Higher than 1 favors the preferentially damaged topology.
    stat, pval = safe_wilcoxon_greater(
        ratio,
        np.ones_like(ratio)
    )
    rb = matched_rank_biserial(
        ratio,
        np.ones_like(ratio)
    )

    ratio_mean, ratio_lo, ratio_hi = t_ci(ratio)

    size_eff_rows.append({
        "graph": graph,
        "independent_trials": len(g),
        "mean_pref_over_size_efficiency": ratio_mean,
        "ratio_ci95_low": ratio_lo,
        "ratio_ci95_high": ratio_hi,
        "wilcoxon_stat": stat,
        "p_one_sided_pref_gt_size": pval,
        "rank_biserial_effect": rb,
    })

size_eff_summary = pd.DataFrame(size_eff_rows)
size_eff_summary["p_bh"] = bh_adjust(
    size_eff_summary["p_one_sided_pref_gt_size"].to_numpy()
)
size_eff_summary["significant_fdr_005"] = (
    size_eff_summary["p_bh"] < ALPHA
)

size_eff_summary.to_csv(
    CONFIRM_DIR / "global_efficiency_size_matched_summary.csv",
    index=False
)

# -----------------------------------------------------------------------------
# EFFECT SIZES FOR THE PRIMARY HITTING-TIME ANALYSIS
# -----------------------------------------------------------------------------

primary_effect_rows = []

primary_trial_scores = pd.read_csv(
    RESULTS_DIR / "trial_sustained_scores.csv"
)

for graph in sorted(primary_trial_scores["graph"].unique()):
    p = primary_trial_scores[
        (primary_trial_scores["graph"] == graph) &
        (primary_trial_scores["damage_type"] == "preferential")
    ][["trial", "sustained_af"]].rename(columns={"sustained_af": "pref"})

    r = primary_trial_scores[
        (primary_trial_scores["graph"] == graph) &
        (primary_trial_scores["damage_type"] == "random")
    ][["trial", "sustained_af"]].rename(columns={"sustained_af": "random"})

    pair = p.merge(r, on="trial", how="inner")
    rb = matched_rank_biserial(pair["pref"], pair["random"])

    delta = pair["pref"] - pair["random"]
    dmean, dlo, dhi = t_ci(delta)

    primary_effect_rows.append({
        "graph": graph,
        "comparison": "preferential_vs_random_sustained_AF",
        "n_pairs": len(pair),
        "mean_difference": dmean,
        "difference_ci95_low": dlo,
        "difference_ci95_high": dhi,
        "rank_biserial_effect": rb,
    })

size_pair_trial = (
    size_pair_primary
    .groupby(["graph", "trial"], as_index=False)
    .agg(
        pref_over_size_ht=("pref_over_size_ht", "mean")
    )
)

for graph, g in size_pair_trial.groupby("graph"):
    ratio = g["pref_over_size_ht"].to_numpy(dtype=float)

    # Benefit = 1 - pref/size, positive means preferential is better.
    benefit = 1.0 - ratio
    bmean, blo, bhi = t_ci(benefit)

    # Compare benefit against zero at the independent-trial level.
    rb = matched_rank_biserial(
        benefit,
        np.zeros_like(benefit)
    )

    primary_effect_rows.append({
        "graph": graph,
        "comparison": "preferential_vs_size_matched_hitting_time",
        "n_pairs": len(g),
        "mean_difference": bmean,
        "difference_ci95_low": blo,
        "difference_ci95_high": bhi,
        "rank_biserial_effect": rb,
    })

primary_effects = pd.DataFrame(primary_effect_rows)
primary_effects.to_csv(
    CONFIRM_DIR / "primary_hitting_time_effect_sizes.csv",
    index=False
)

# -----------------------------------------------------------------------------
# TRANSPARENT CLASSIFICATION
#
# "Controlled AF (HT)" requires:
#   - at least 3 damage levels with >=2% mean HT improvement,
#   - preferential > random sustained AF after BH-FDR,
#   - preferential HT < size-matched HT after BH-FDR,
#   - >=80% mean LCC retention at best damage.
#
# Global efficiency is reported as an INDEPENDENT CONFIRMATORY CHECK.
# It does not overwrite the primary hitting-time classification.
# -----------------------------------------------------------------------------

classification = network_summary.copy()

classification["ht_pref_beats_random"] = (
    classification["pref_vs_random_p_bh"] < ALPHA
)
classification["ht_beats_size_control"] = (
    classification["size_control_p_bh"] < ALPHA
)
classification["sustained_levels_ok"] = (
    classification["damage_levels_ge_2pct"] >= MIN_IMPROVEMENT_LEVELS
)
classification["lcc_retention_ok"] = (
    classification["mean_lcc_fraction_at_best"] >= MIN_LCC_FRACTION
)

def classify_ht(row):
    core = (
        row["sustained_levels_ok"] and
        row["lcc_retention_ok"]
    )

    if (
        core and
        row["ht_pref_beats_random"] and
        row["ht_beats_size_control"]
    ):
        return "Controlled antifragile transport"

    if core and (
        row["ht_pref_beats_random"] or
        row["ht_beats_size_control"]
    ):
        return "Partial / control-dependent evidence"

    return "Unsupported under strict controls"

classification["primary_ht_classification"] = classification.apply(
    classify_ht, axis=1
)

classification = classification.merge(
    pvr_eff[
        [
            "graph",
            "mean_sustained_efficiency_preferential",
            "mean_sustained_efficiency_random",
            "p_bh",
            "rank_biserial_effect",
        ]
    ].rename(
        columns={
            "p_bh": "global_eff_pref_vs_random_p_bh",
            "rank_biserial_effect":
                "global_eff_pref_vs_random_rank_biserial",
        }
    ),
    on="graph",
    how="left"
)

classification = classification.merge(
    size_eff_summary[
        [
            "graph",
            "mean_pref_over_size_efficiency",
            "p_bh",
            "rank_biserial_effect",
        ]
    ].rename(
        columns={
            "p_bh": "global_eff_size_control_p_bh",
            "rank_biserial_effect":
                "global_eff_size_control_rank_biserial",
        }
    ),
    on="graph",
    how="left"
)

classification["global_efficiency_confirms"] = (
    (classification["global_eff_pref_vs_random_p_bh"] < ALPHA) &
    (classification["global_eff_size_control_p_bh"] < ALPHA) &
    (
        classification["mean_sustained_efficiency_preferential"] >
        classification["mean_sustained_efficiency_random"]
    ) &
    (classification["mean_pref_over_size_efficiency"] > 1.0)
)

classification.to_csv(
    CONFIRM_DIR / "benchmark_classification.csv",
    index=False
)

# -----------------------------------------------------------------------------
# FIGURES
# -----------------------------------------------------------------------------

for name in sorted(eff_curve["graph"].unique()):
    sub = eff_curve[eff_curve["graph"] == name]

    plt.figure(figsize=(6.2, 4.2))

    for dtype in ("preferential", "random"):
        s = sub[sub["damage_type"] == dtype].sort_values("damage_frac")
        if s.empty:
            continue

        x = 100.0 * s["damage_frac"].to_numpy()
        y = s["mean_relative_global_efficiency"].to_numpy()
        lo = s["ci95_low"].to_numpy()
        hi = s["ci95_high"].to_numpy()

        plt.plot(x, y, marker="o", label=dtype.capitalize())
        plt.fill_between(x, lo, hi, alpha=0.18)

    plt.axhline(1.0, linestyle="--", linewidth=1)
    plt.xlabel("Edge damage (%)")
    plt.ylabel("Relative global efficiency")
    plt.title(name.replace("_", " "))
    plt.legend()
    plt.tight_layout()
    plt.savefig(
        CONFIRM_FIG_DIR /
        f"{name.lower()}_global_efficiency_pref_vs_random.pdf",
        bbox_inches="tight"
    )
    plt.close()

# Size-control ratio figure.
plot_groups = []
plot_labels = []

for name in sorted(size_eff_pairs["graph"].unique()):
    vals = size_eff_pairs.loc[
        size_eff_pairs["graph"] == name,
        "pref_over_size_efficiency"
    ].to_numpy()
    plot_groups.append(vals)
    plot_labels.append(name.replace("_", " "))

plt.figure(figsize=(8.0, 4.8))
plt.boxplot(plot_groups, tick_labels=plot_labels, showfliers=False)
plt.axhline(1.0, linestyle="--", linewidth=1)
plt.ylabel("Preferential / size-matched global efficiency")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(
    CONFIRM_FIG_DIR / "global_efficiency_size_control.pdf",
    bbox_inches="tight"
)
plt.close()

# -----------------------------------------------------------------------------
# AUTOMATIC LATEX TABLE
# -----------------------------------------------------------------------------

latex_path = CONFIRM_DIR / "confirmatory_results_auto.tex"

def fmt_p(p):
    if not np.isfinite(p):
        return "--"
    if p < 0.001:
        return f"{p:.2e}"
    return f"{p:.3f}"

lines = [
    r"% Automatically generated. Do not edit by hand.",
    r"\begin{table*}[!t]",
    r"\centering",
    r"\caption{Primary hitting-time controls and confirmatory global-efficiency results.}",
    r"\label{tab:confirmatory}",
    r"\footnotesize",
    r"\setlength{\tabcolsep}{4pt}",
    r"\begin{tabular}{lcccccc}",
    r"\toprule",
    r"Network & HT Class & $p_{\rm PR}^{HT}$ & $p_{\rm size}^{HT}$ & "
    r"$p_{\rm PR}^{E}$ & $p_{\rm size}^{E}$ & $E_{\rm pref}/E_{\rm size}$ \\",
    r"\midrule",
]

for _, r in classification.sort_values("graph").iterrows():
    short_class = {
        "Controlled antifragile transport": "Controlled",
        "Partial / control-dependent evidence": "Partial",
        "Unsupported under strict controls": "Unsupported",
    }[r["primary_ht_classification"]]

    lines.append(
        f"{r['graph'].replace('_', ' ')} & "
        f"{short_class} & "
        f"{fmt_p(r['pref_vs_random_p_bh'])} & "
        f"{fmt_p(r['size_control_p_bh'])} & "
        f"{fmt_p(r['global_eff_pref_vs_random_p_bh'])} & "
        f"{fmt_p(r['global_eff_size_control_p_bh'])} & "
        f"{r['mean_pref_over_size_efficiency']:.3f} \\\\"
    )

lines += [
    r"\bottomrule",
    r"\end{tabular}",
    r"\end{table*}",
]

latex_path.write_text("\n".join(lines), encoding="utf-8")

# -----------------------------------------------------------------------------
# FINAL REPORT
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("PRIMARY HITTING-TIME CLASSIFICATION")
print("=" * 80)

print(
    classification[
        [
            "graph",
            "primary_ht_classification",
            "pref_vs_random_p_bh",
            "size_control_p_bh",
            "mean_lcc_fraction_at_best",
        ]
    ].to_string(index=False)
)

print("\n" + "=" * 80)
print("CONFIRMATORY GLOBAL-EFFICIENCY RESULTS")
print("=" * 80)

print(
    classification[
        [
            "graph",
            "mean_sustained_efficiency_preferential",
            "mean_sustained_efficiency_random",
            "global_eff_pref_vs_random_p_bh",
            "mean_pref_over_size_efficiency",
            "global_eff_size_control_p_bh",
            "global_efficiency_confirms",
        ]
    ].to_string(index=False)
)

print("\nSaved confirmatory outputs:")
for p in [
    CONFIRM_DIR / "baseline_global_efficiency.csv",
    CONFIRM_DIR / "global_efficiency_raw.csv",
    CONFIRM_DIR / "global_efficiency_curve_statistics.csv",
    CONFIRM_DIR / "trial_sustained_global_efficiency.csv",
    CONFIRM_DIR / "global_efficiency_preferential_vs_random.csv",
    CONFIRM_DIR / "global_efficiency_size_matched_pair_level.csv",
    CONFIRM_DIR / "global_efficiency_size_matched_summary.csv",
    CONFIRM_DIR / "primary_hitting_time_effect_sizes.csv",
    CONFIRM_DIR / "benchmark_classification.csv",
    CONFIRM_DIR / "confirmatory_results_auto.tex",
]:
    print(" ", p)

print("\nFigures:")
print(" ", CONFIRM_FIG_DIR)
print("\nCONFIRMATORY BENCHMARK ANALYSIS COMPLETE")

CONFIRMATORY BENCHMARK ANALYSIS
Synthetic networks: NONE
Hardcoded empirical outcomes: NONE
Global-efficiency source samples per graph state: 24
Confirmatory damage fractions: [0.1, 0.2, 0.3, 0.4]
Resuming existing confirmatory results: /content/drive/MyDrive/Markov_Antifragility_IEEE/results_real_only/confirmatory_benchmarks/global_efficiency_raw.csv

[1/6] ca_GrQc: confirmatory global efficiency, trials=20
  completed trial  1/20
  completed trial  2/20
  completed trial  3/20
  completed trial  4/20
  completed trial  5/20
  completed trial  6/20
  completed trial  7/20
  completed trial  8/20
  completed trial  9/20
  completed trial 10/20
  completed trial 11/20
  completed trial 12/20
  completed trial 13/20
  completed trial 14/20
  completed trial 15/20
  completed trial 16/20
  completed trial 17/20
  completed trial 18/20
  completed trial 19/20
  completed trial 20/20

[2/6] ca_HepTh: confirmatory global efficiency, trials=15
  completed trial  1/15
  completed trial  2/15
 

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

# Reuse the same output directory created by the main experiment.
STRUCT_PATH = RESULTS_DIR / "network_structure.csv"
SUMMARY_PATH = RESULTS_DIR / "network_summary.csv"

struct = pd.read_csv(STRUCT_PATH)
summary = pd.read_csv(SUMMARY_PATH)

# Merge only observed network-level quantities.
df = struct.merge(
    summary[
        [
            "graph",
            "mean_sustained_af_preferential",
            "mean_sustained_af_random",
            "max_mean_improvement_pct",
            "mean_lcc_fraction_at_best",
        ]
    ],
    on="graph",
    how="inner",
    validate="one_to_one",
)

if len(df) != 6:
    raise ValueError(f"Expected 6 real networks, found {len(df)}.")

target = "mean_sustained_af_preferential"

features = [
    "modularity",
    "density",
    "mean_degree",
    "degree_cv",
    "average_clustering",
    "transitivity",
    "assortativity",
    "communities",
    "nodes",
    "edges",
]

def bh_adjust(pvalues):
    p = np.asarray(pvalues, dtype=float)
    m = len(p)
    order = np.argsort(p)
    ranked = p[order]
    adj = ranked * m / np.arange(1, m + 1)
    adj = np.minimum.accumulate(adj[::-1])[::-1]
    adj = np.clip(adj, 0.0, 1.0)
    out = np.empty(m, dtype=float)
    out[order] = adj
    return out

rows = []

for feature in features:
    rho, p = spearmanr(df[feature], df[target])

    rows.append(
        {
            "feature": feature,
            "spearman_rho": float(rho),
            "abs_rho": abs(float(rho)),
            "p_value": float(p),
            "direction": (
                "positive" if rho > 0
                else "negative" if rho < 0
                else "none"
            ),
        }
    )

results = pd.DataFrame(rows)
results["p_bh"] = bh_adjust(results["p_value"].to_numpy())

# Sort by association magnitude, NOT by p-value.
results = results.sort_values(
    ["abs_rho", "feature"],
    ascending=[False, True],
).reset_index(drop=True)

# Descriptive interpretation only.
def magnitude(r):
    a = abs(r)
    if a >= 0.80:
        return "very strong"
    if a >= 0.60:
        return "strong"
    if a >= 0.40:
        return "moderate"
    if a >= 0.20:
        return "weak"
    return "very weak"

results["association_magnitude"] = results["spearman_rho"].apply(magnitude)

OUT_DIR = RESULTS_DIR / "exploratory_structural_correlates"
OUT_DIR.mkdir(parents=True, exist_ok=True)

csv_path = OUT_DIR / "structural_correlates_spearman.csv"
results.to_csv(csv_path, index=False)

# Also save the exact six-network merged table used in the analysis.
data_path = OUT_DIR / "structural_correlates_input.csv"
df.to_csv(data_path, index=False)

# Automatic LaTeX table.
latex_path = OUT_DIR / "structural_correlates_auto.tex"

def fmt_p(p):
    if p < 0.001:
        return f"{p:.2e}"
    return f"{p:.3f}"

latex = [
    r"% Automatically generated exploratory structural-correlation table.",
    r"\begin{table}[!t]",
    r"\centering",
    r"\caption{Exploratory Spearman Associations with Sustained Preferential Antifragility ($n=6$ Real Networks)}",
    r"\label{tab:structural_correlates}",
    r"\footnotesize",
    r"\begin{tabular}{lrrr}",
    r"\toprule",
    r"Structural Feature & $\rho$ & $p$ & $p_{\rm BH}$ \\",
    r"\midrule",
]

for _, r in results.iterrows():
    label = r["feature"].replace("_", " ").title()
    latex.append(
        f"{label} & {r['spearman_rho']:.3f} & "
        f"{fmt_p(r['p_value'])} & {fmt_p(r['p_bh'])} \\\\"
    )

latex += [
    r"\bottomrule",
    r"\end{tabular}",
    r"\end{table}",
]

latex_path.write_text("\n".join(latex), encoding="utf-8")

print("=" * 80)
print("EXPLORATORY STRUCTURAL-CORRELATE ANALYSIS")
print("=" * 80)
print("Networks:", len(df))
print("Target:", target)
print()
print(
    results[
        [
            "feature",
            "spearman_rho",
            "association_magnitude",
            "p_value",
            "p_bh",
        ]
    ].to_string(index=False)
)

print("\nIMPORTANT INTERPRETATION:")
print(
    "Because n=6, these correlations are exploratory only. "
    "Do not interpret any structural feature as causal."
)
print(
    "Use |rho| to identify candidate structural explanations for future study; "
    "use p-values only as secondary evidence."
)

print("\nSaved:")
print(" ", data_path)
print(" ", csv_path)
print(" ", latex_path)
print("\nEXPLORATORY STRUCTURAL-CORRELATE ANALYSIS COMPLETE")

EXPLORATORY STRUCTURAL-CORRELATE ANALYSIS
Networks: 6
Target: mean_sustained_af_preferential

           feature  spearman_rho association_magnitude  p_value     p_bh
average_clustering     -0.771429                strong 0.072397 0.520000
      transitivity     -0.657143                strong 0.156175 0.520000
     assortativity     -0.600000                strong 0.208000 0.520000
        modularity     -0.600000                strong 0.208000 0.520000
         degree_cv      0.542857              moderate 0.265703 0.531405
       communities     -0.371429                  weak 0.468478 0.780797
           density      0.200000                  weak 0.704000 0.871743
             nodes     -0.200000                  weak 0.704000 0.871743
             edges      0.085714             very weak 0.871743 0.871743
       mean_degree      0.085714             very weak 0.871743 0.871743

IMPORTANT INTERPRETATION:
Because n=6, these correlations are exploratory only. Do not interpret any s

In [ ]:
from pathlib import Path
import sys
import subprocess

# Install PyMuPDF only if unavailable.
try:
    import fitz  # PyMuPDF
except ImportError:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "pymupdf"]
    )
    import fitz

# Exact confirmatory figure directory used by the benchmark notebook.
FIG_DIR = Path(
    "/content/drive/MyDrive/Markov_Antifragility_IEEE/"
    "results_real_only/confirmatory_benchmarks/figures"
)

if not FIG_DIR.exists():
    raise FileNotFoundError(
        f"Confirmatory figure directory not found:\n{FIG_DIR}"
    )

pdf_files = sorted(FIG_DIR.glob("*.pdf"))

if not pdf_files:
    raise FileNotFoundError(
        f"No PDF figures found in:\n{FIG_DIR}"
    )

print("=" * 80)
print("CONFIRMATORY FIGURE CONVERSION — PDF TO PNG")
print("=" * 80)
print(f"Source/output directory: {FIG_DIR}")
print(f"PDF files found: {len(pdf_files)}")
print()

converted = []
failed = []

# 300 dpi equivalent scaling from PDF's nominal 72 dpi.
zoom = 300.0 / 72.0
matrix = fitz.Matrix(zoom, zoom)

for pdf_path in pdf_files:
    png_path = pdf_path.with_suffix(".png")

    try:
        with fitz.open(pdf_path) as doc:
            if doc.page_count < 1:
                raise RuntimeError("PDF contains no pages")

            # Figures generated by matplotlib are one-page PDFs.
            page = doc.load_page(0)
            pix = page.get_pixmap(
                matrix=matrix,
                alpha=False
            )
            pix.save(str(png_path))

        converted.append(png_path)
        print(f"CONVERTED: {pdf_path.name}")
        print(f"        -> {png_path.name}")

    except Exception as exc:
        failed.append((pdf_path, str(exc)))
        print(f"FAILED: {pdf_path.name}: {exc}")

# -------------------------------------------------------------------------
# Validate the exact confirmatory filenames used by the revised manuscript.
# -------------------------------------------------------------------------

required_pngs = [
    "ca_grqc_global_efficiency_pref_vs_random.png",
    "ca_hepth_global_efficiency_pref_vs_random.png",
    "facebook_combined_global_efficiency_pref_vs_random.png",
    "email_eu_core_global_efficiency_pref_vs_random.png",
    "wiki_vote_global_efficiency_pref_vs_random.png",
    "collegemsg_global_efficiency_pref_vs_random.png",
    "global_efficiency_size_control.png",
]

print("\n" + "=" * 80)
print("MANUSCRIPT FILENAME VALIDATION")
print("=" * 80)

missing = []

for filename in required_pngs:
    path = FIG_DIR / filename
    if path.exists() and path.stat().st_size > 0:
        print(f"PASS — {filename}")
    else:
        print(f"MISSING — {filename}")
        missing.append(filename)

print("\n" + "=" * 80)
print("CONVERSION SUMMARY")
print("=" * 80)
print(f"Converted successfully: {len(converted)}")
print(f"Conversion failures:    {len(failed)}")
print(f"Required PNGs missing:  {len(missing)}")

if failed:
    print("\nFailures:")
    for path, err in failed:
        print(f"  {path.name}: {err}")

if missing:
    raise RuntimeError(
        "Conversion completed, but one or more manuscript-required PNG files "
        "are still missing:\n  " + "\n  ".join(missing)
    )

print("\nALL CONFIRMATORY PNG FIGURES ARE PRESENT.")
print("You can now copy/move these PNG files into the same Overleaf/GitHub")
print("directory as the LaTeX manuscript without changing any filenames.")

CONFIRMATORY FIGURE CONVERSION — PDF TO PNG
Source/output directory: /content/drive/MyDrive/Markov_Antifragility_IEEE/results_real_only/confirmatory_benchmarks/figures
PDF files found: 7

CONVERTED: ca_grqc_global_efficiency_pref_vs_random.pdf
        -> ca_grqc_global_efficiency_pref_vs_random.png
CONVERTED: ca_hepth_global_efficiency_pref_vs_random.pdf
        -> ca_hepth_global_efficiency_pref_vs_random.png
CONVERTED: collegemsg_global_efficiency_pref_vs_random.pdf
        -> collegemsg_global_efficiency_pref_vs_random.png
CONVERTED: email_eu_core_global_efficiency_pref_vs_random.pdf
        -> email_eu_core_global_efficiency_pref_vs_random.png
CONVERTED: facebook_combined_global_efficiency_pref_vs_random.pdf
        -> facebook_combined_global_efficiency_pref_vs_random.png
CONVERTED: global_efficiency_size_control.pdf
        -> global_efficiency_size_control.png
CONVERTED: wiki_vote_global_efficiency_pref_vs_random.pdf
        -> wiki_vote_global_efficiency_pref_vs_random.png

MAN

In [ ]:
from pathlib import Path
import shutil
import sys
import subprocess

# -------------------------------------------------------------------------
# DIRECTORIES
# -------------------------------------------------------------------------

BASE_DIR = Path("/content/drive/MyDrive/Markov_Antifragility_IEEE")

PRIMARY_FIG_DIR = (
    BASE_DIR
    / "results_real_only"
    / "figures"
)

CONFIRM_FIG_DIR = (
    BASE_DIR
    / "results_real_only"
    / "confirmatory_benchmarks"
    / "figures"
)

ARTICLE_FIG_DIR = BASE_DIR / "article_figures"

ARTICLE_FIG_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------------------------------
# EXACT FINAL MANUSCRIPT FIGURES
# -------------------------------------------------------------------------

PRIMARY_FIGURES = [
    "ca_grqc_preferential_vs_random.png",
    "ca_hepth_preferential_vs_random.png",
    "facebook_combined_preferential_vs_random.png",
    "email_eu_core_preferential_vs_random.png",
    "wiki_vote_preferential_vs_random.png",
    "collegemsg_preferential_vs_random.png",
    "size_matched_control_real_networks.png",
    "lcc_retention_all_real_networks.png",
    "modularity_vs_sustained_af_real_only.png",
]

CONFIRMATORY_FIGURES = [
    "global_efficiency_size_control.png",
    "ca_grqc_global_efficiency_pref_vs_random.png",
    "ca_hepth_global_efficiency_pref_vs_random.png",
    "facebook_combined_global_efficiency_pref_vs_random.png",
    "email_eu_core_global_efficiency_pref_vs_random.png",
    "wiki_vote_global_efficiency_pref_vs_random.png",
    "collegemsg_global_efficiency_pref_vs_random.png",
]

ALL_REQUIRED = PRIMARY_FIGURES + CONFIRMATORY_FIGURES

print("=" * 80)
print("FINAL ARTICLE FIGURE COLLECTION")
print("=" * 80)
print(f"Primary source:      {PRIMARY_FIG_DIR}")
print(f"Confirmatory source: {CONFIRM_FIG_DIR}")
print(f"Final article dir:   {ARTICLE_FIG_DIR}")
print()

# -------------------------------------------------------------------------
# PDF -> PNG FALLBACK FOR CONFIRMATORY FIGURES
# -------------------------------------------------------------------------

try:
    import fitz  # PyMuPDF
except ImportError:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "pymupdf"]
    )
    import fitz

def convert_pdf_to_png(pdf_path: Path, png_path: Path):
    """Convert first page of a one-page matplotlib PDF to a 300-DPI PNG."""
    zoom = 300.0 / 72.0
    matrix = fitz.Matrix(zoom, zoom)

    with fitz.open(pdf_path) as doc:
        if doc.page_count < 1:
            raise RuntimeError(f"No pages in {pdf_path.name}")

        page = doc.load_page(0)
        pix = page.get_pixmap(matrix=matrix, alpha=False)
        pix.save(str(png_path))

# -------------------------------------------------------------------------
# CLEAN OLD CONTENTS FROM FINAL ARTICLE DIRECTORY
# -------------------------------------------------------------------------

for item in ARTICLE_FIG_DIR.iterdir():
    if item.is_file():
        item.unlink()

print("Cleared old files from article_figures/.\n")

# -------------------------------------------------------------------------
# COPY PRIMARY FIGURES
# -------------------------------------------------------------------------

missing_primary = []

print("PRIMARY STUDY FIGURES")
print("-" * 80)

for filename in PRIMARY_FIGURES:
    src = PRIMARY_FIG_DIR / filename
    dst = ARTICLE_FIG_DIR / filename

    if src.exists() and src.stat().st_size > 0:
        shutil.copy2(src, dst)
        print(f"COPIED  {filename}")
    else:
        missing_primary.append(filename)
        print(f"MISSING {filename}")

# -------------------------------------------------------------------------
# COPY / CONVERT CONFIRMATORY FIGURES
# -------------------------------------------------------------------------

missing_confirm = []

print("\nCONFIRMATORY FIGURES")
print("-" * 80)

for filename in CONFIRMATORY_FIGURES:
    png_src = CONFIRM_FIG_DIR / filename
    pdf_src = CONFIRM_FIG_DIR / Path(filename).with_suffix(".pdf").name
    dst = ARTICLE_FIG_DIR / filename

    # Preferred: existing PNG.
    if png_src.exists() and png_src.stat().st_size > 0:
        shutil.copy2(png_src, dst)
        print(f"COPIED     {filename}")
        continue

    # Fallback: convert existing PDF to PNG.
    if pdf_src.exists() and pdf_src.stat().st_size > 0:
        temp_png = CONFIRM_FIG_DIR / filename
        convert_pdf_to_png(pdf_src, temp_png)
        shutil.copy2(temp_png, dst)
        print(f"CONVERTED  {pdf_src.name}")
        print(f"        -> {filename}")
        continue

    missing_confirm.append(filename)
    print(f"MISSING    {filename}")
    print(f"  Also checked PDF fallback: {pdf_src.name}")

# -------------------------------------------------------------------------
# FINAL VALIDATION
# -------------------------------------------------------------------------

print("\n" + "=" * 80)
print("FINAL ARTICLE DIRECTORY VALIDATION")
print("=" * 80)

missing_final = []

for filename in ALL_REQUIRED:
    path = ARTICLE_FIG_DIR / filename

    if path.exists() and path.stat().st_size > 0:
        print(f"PASS — {filename}")
    else:
        print(f"FAIL — {filename}")
        missing_final.append(filename)

# Confirm PNG-only.
non_png = [
    p.name
    for p in ARTICLE_FIG_DIR.iterdir()
    if p.is_file() and p.suffix.lower() != ".png"
]

print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)
print(f"Required figures:       {len(ALL_REQUIRED)}")
print(f"Files in article dir:   {len(list(ARTICLE_FIG_DIR.glob('*')))}")
print(f"Missing final figures:  {len(missing_final)}")
print(f"Non-PNG files present:  {len(non_png)}")

if non_png:
    print("\nUnexpected non-PNG files:")
    for f in non_png:
        print(" ", f)

if missing_final:
    raise RuntimeError(
        "The final article figure directory is incomplete.\n"
        "Missing:\n  " + "\n  ".join(missing_final)
    )

if non_png:
    raise RuntimeError(
        "The final article figure directory contains non-PNG files."
    )

print("\nSUCCESS.")
print("\nUSE ONLY THIS DIRECTORY FOR THE ARTICLE:")
print(ARTICLE_FIG_DIR)
print("\nIt contains exactly these manuscript figures:\n")

for i, filename in enumerate(ALL_REQUIRED, start=1):
    print(f"{i:02d}. {filename}")

FINAL ARTICLE FIGURE COLLECTION
Primary source:      /content/drive/MyDrive/Markov_Antifragility_IEEE/results_real_only/figures
Confirmatory source: /content/drive/MyDrive/Markov_Antifragility_IEEE/results_real_only/confirmatory_benchmarks/figures
Final article dir:   /content/drive/MyDrive/Markov_Antifragility_IEEE/article_figures

Cleared old files from article_figures/.

PRIMARY STUDY FIGURES
--------------------------------------------------------------------------------
COPIED  ca_grqc_preferential_vs_random.png
COPIED  ca_hepth_preferential_vs_random.png
COPIED  facebook_combined_preferential_vs_random.png
COPIED  email_eu_core_preferential_vs_random.png
COPIED  wiki_vote_preferential_vs_random.png
COPIED  collegemsg_preferential_vs_random.png
COPIED  size_matched_control_real_networks.png
COPIED  lcc_retention_all_real_networks.png
COPIED  modularity_vs_sustained_af_real_only.png

CONFIRMATORY FIGURES
------------------------------------------------------------------------------

In [ ]:
import os, sys, json, math, time, hashlib, random, re, subprocess
from pathlib import Path
from collections import Counter

required = {"datasets":"datasets","networkx":"networkx","numpy":"numpy","pandas":"pandas","scipy":"scipy","matplotlib":"matplotlib"}
for module, package in required.items():
    try: __import__(module)
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install","-q",package])

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from datasets import load_dataset
from scipy import sparse
from scipy.sparse.linalg import splu
from scipy.stats import wilcoxon, t, rankdata, binomtest

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    print('Google Drive mount skipped; assuming Drive is already mounted.')

BASE_DIR = Path('/content/drive/MyDrive/Markov_Antifragility_IEEE')
OUT_DIR = BASE_DIR / 'ai_domain_validation_exgentic'
FIG_DIR = OUT_DIR / 'figures_png'
CHECKPOINT_DIR = OUT_DIR / 'checkpoints'
for p in (OUT_DIR, FIG_DIR, CHECKPOINT_DIR): p.mkdir(parents=True, exist_ok=True)

DATASET_ID='Exgentic/agent-llm-traces-v2'
DATASET_URL='https://huggingface.co/datasets/Exgentic/agent-llm-traces-v2'
DATASET_LICENSE='CDLA-Permissive-2.0'
GRAPH_NAME='Exgentic_AgentTool'
DAMAGE_FRACTIONS=np.arange(0.05,0.451,0.05)
SIZE_CONTROL_FRACTIONS=np.array([0.10,0.20,0.30,0.40])
N_TRIALS=30
HUTCHINSON_PROBES=24
GLOBAL_EFFICIENCY_SOURCES=24
IMPROVEMENT_THRESHOLD=0.98
ALPHA=0.05
MIN_LCC_FRACTION=0.80
MIN_IMPROVEMENT_LEVELS=3
MASTER_SEED=20260814


def stable_seed(*parts):
    h=hashlib.sha256('||'.join(map(str,parts)).encode()).digest()
    return int.from_bytes(h[:8],'little')%(2**32-1)

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''): h.update(block)
    return h.hexdigest()

def save_png(stem):
    path=FIG_DIR/f'{stem}.png'; plt.savefig(path,dpi=300,bbox_inches='tight'); plt.close(); return path

def t_ci(x,alpha=0.05):
    x=np.asarray(x,float); x=x[np.isfinite(x)]; n=len(x)
    if n==0: return np.nan,np.nan,np.nan
    m=float(x.mean())
    if n==1: return m,np.nan,np.nan
    se=float(x.std(ddof=1)/np.sqrt(n)); q=float(t.ppf(1-alpha/2,n-1)); return m,m-q*se,m+q*se

def matched_rb(diff):
    diff=np.asarray(diff,float); diff=diff[np.isfinite(diff)]; diff=diff[diff!=0]
    if len(diff)==0: return 0.0
    r=rankdata(np.abs(diff),method='average'); wp=r[diff>0].sum(); wn=r[diff<0].sum(); return float((wp-wn)/(wp+wn))

def safe_wilcoxon(diff,alternative='greater'):
    diff=np.asarray(diff,float); diff=diff[np.isfinite(diff)]
    if len(diff)==0: return np.nan,np.nan
    if np.allclose(diff,0): return 0.0,1.0
    try:
        s,p=wilcoxon(diff,alternative=alternative,zero_method='wilcox'); return float(s),float(p)
    except Exception:
        nz=diff[diff!=0]
        if len(nz)==0: return 0.0,1.0
        if alternative=='greater': k=int((nz>0).sum())
        else: k=int((nz<0).sum())
        return np.nan,float(binomtest(k,len(nz),0.5,alternative='greater').pvalue)

def flatten_dict(d,prefix=''):
    out={}
    if isinstance(d,dict):
        for k,v in d.items():
            key=f'{prefix}.{k}' if prefix else str(k)
            if isinstance(v,dict): out.update(flatten_dict(v,key))
            else: out[key]=v
    return out

def maybe_json(v):
    if isinstance(v,(dict,list)): return v
    if isinstance(v,str):
        s=v.strip()
        if s and s[0] in '[{':
            try: return json.loads(s)
            except Exception: pass
    return v

def recursive_tool_calls(obj):
    names=[]; obj=maybe_json(obj)
    if isinstance(obj,dict):
        typ=str(obj.get('type','')).lower()
        if typ in {'tool_call','function_call','tool_use'}:
            name=obj.get('name') or obj.get('tool_name') or obj.get('function')
            if isinstance(name,dict): name=name.get('name')
            if name: names.append(str(name))
        fn=obj.get('function')
        if isinstance(fn,dict) and fn.get('name') and (typ in {'tool_call','function_call'} or 'arguments' in fn): names.append(str(fn['name']))
        if obj.get('tool_name') and ('arguments' in obj or 'args' in obj or 'result' in obj): names.append(str(obj['tool_name']))
        for v in obj.values(): names.extend(recursive_tool_calls(v))
    elif isinstance(obj,list):
        for v in obj: names.extend(recursive_tool_calls(v))
    return names

def normalize_tool_name(name):
    name=re.sub(r'\s+',' ',str(name).strip()); return name[:300] if name else None

def normalize_model(model):
    if isinstance(model,list): return [str(x).strip() for x in model if str(x).strip()]
    return [str(model).strip()] if model is not None and str(model).strip() else []

print('='*88); print('1. LOADING REAL AGENT EXECUTION TRACES'); print('='*88); print(DATASET_ID)
ds=load_dataset(DATASET_ID,split='train',streaming=True)
G=nx.Graph(); node_obs=Counter(); trace_count=span_count=tool_call_count=trace_with_tools=0

def add_edge(a,b,rel):
    if not a or not b or a==b: return
    u,v=sorted((str(a),str(b))); node_obs[u]+=1; node_obs[v]+=1
    if not G.has_edge(u,v): G.add_edge(u,v,observed_count=0,relationship_types=set())
    G[u][v]['observed_count']+=1; G[u][v]['relationship_types'].add(rel)

for row in ds:
    trace_count+=1
    benchmark=str(row.get('benchmark','unknown')).strip(); harness=str(row.get('harness','unknown')).strip()
    bn=f'benchmark::{benchmark}'; hn=f'harness::{harness}'
    G.add_node(bn,node_type='benchmark',label=benchmark); G.add_node(hn,node_type='harness',label=harness); add_edge(bn,hn,'benchmark_harness')
    model_nodes=[]
    for model in normalize_model(row.get('models')):
        mn=f'model::{model}'; model_nodes.append(mn); G.add_node(mn,node_type='model',label=model); add_edge(hn,mn,'harness_model')
    seq=[]
    for span in (row.get('spans') or []):
        span_count+=1; attrs=span.get('attributes',{}) if isinstance(span,dict) else {}; flat=flatten_dict(attrs) if isinstance(attrs,dict) else {}
        for key in ('gen_ai.request.model','gen_ai.response.model','gen_ai.request.model_name','gen_ai.response.model_name'):
            if key in flat and flat[key]:
                for model in normalize_model(flat[key]):
                    mn=f'model::{model}'
                    if mn not in model_nodes: model_nodes.append(mn)
                    G.add_node(mn,node_type='model',label=model); add_edge(hn,mn,'harness_model')
        tools=[]
        for key in ('gen_ai.output.messages','gen_ai.output.message','output.messages','output'):
            if key in flat: tools.extend(recursive_tool_calls(flat[key]))
        for key in ('gen_ai.tool.name','tool.name','tool_name'):
            if key in flat and flat[key]: tools.append(str(flat[key]))
        if isinstance(span,dict):
            sname=str(span.get('name','')); low=sname.lower()
            if 'tool' in low and ':' in sname:
                cand=sname.split(':',1)[1].strip()
                if cand: tools.append(cand)
        for tool in tools:
            tool=normalize_tool_name(tool)
            if not tool: continue
            tn=f'tool::{tool}'; G.add_node(tn,node_type='tool',label=tool); tool_call_count+=1; seq.append(tn)
            for parent in (model_nodes if model_nodes else [hn]): add_edge(parent,tn,'model_tool')
    if seq:
        trace_with_tools+=1
        for a,b in zip(seq[:-1],seq[1:]):
            if a!=b: add_edge(a,b,'tool_transition')
    if trace_count%500==0: print(f'{trace_count:,} traces | {G.number_of_nodes():,} nodes | {G.number_of_edges():,} edges | {tool_call_count:,} tool calls')

G.remove_edges_from(nx.selfloop_edges(G)); G.remove_nodes_from(list(nx.isolates(G)))
if G.number_of_nodes()<10 or G.number_of_edges()<10: raise RuntimeError('Observed AI interaction graph unexpectedly small.')
G0=G.subgraph(max(nx.connected_components(G),key=len)).copy()
print('Raw graph:',G.number_of_nodes(),G.number_of_edges(),' LCC:',G0.number_of_nodes(),G0.number_of_edges())

node_path=OUT_DIR/'exgentic_agent_tool_nodes.csv'; edge_path=OUT_DIR/'exgentic_agent_tool_edges.csv'
pd.DataFrame([{'node':n,'node_type':d.get('node_type','unknown'),'label':d.get('label',''),'observation_count':int(node_obs.get(n,0))} for n,d in G.nodes(data=True)]).to_csv(node_path,index=False)
pd.DataFrame([{'source':u,'target':v,'observed_count':int(d.get('observed_count',1)),'relationship_types':';'.join(sorted(d.get('relationship_types',[])))} for u,v,d in G.edges(data=True)]).to_csv(edge_path,index=False)
pd.DataFrame([{'graph':GRAPH_NAME,'dataset_id':DATASET_ID,'dataset_url':DATASET_URL,'license':DATASET_LICENSE,'retrieved_runtime':pd.Timestamp.utcnow().isoformat(),'traces_processed':trace_count,'spans_processed':span_count,'traces_with_tools':trace_with_tools,'observed_tool_calls':tool_call_count,'raw_graph_nodes':G.number_of_nodes(),'raw_graph_edges':G.number_of_edges(),'baseline_lcc_nodes':G0.number_of_nodes(),'baseline_lcc_edges':G0.number_of_edges(),'node_file_sha256':sha256_file(node_path),'edge_file_sha256':sha256_file(edge_path),'graph_construction':'Observed benchmark-harness, harness-model, model-tool, and consecutive tool-transition relations; simple undirected graph; baseline LCC retained; no synthetic nodes or edges.'}]).to_csv(OUT_DIR/'dataset_provenance.csv',index=False)

deg=np.array([d for _,d in G0.degree()],float); mean_degree=float(deg.mean()); degree_cv=float(deg.std(ddof=1)/mean_degree) if mean_degree>0 else np.nan
try: communities=list(nx.community.louvain_communities(G0,seed=stable_seed(GRAPH_NAME,'louvain')))
except Exception: communities=list(nx.community.greedy_modularity_communities(G0))
modularity=float(nx.community.modularity(G0,communities)); nt=Counter(d.get('node_type','unknown') for _,d in G0.nodes(data=True))
structure=pd.DataFrame([{'graph':GRAPH_NAME,'nodes':G0.number_of_nodes(),'edges':G0.number_of_edges(),'density':nx.density(G0),'mean_degree':mean_degree,'degree_cv':degree_cv,'average_clustering':nx.average_clustering(G0),'transitivity':nx.transitivity(G0),'assortativity':nx.degree_assortativity_coefficient(G0),'modularity':modularity,'communities':len(communities),'tool_nodes':nt.get('tool',0),'model_nodes':nt.get('model',0),'harness_nodes':nt.get('harness',0),'benchmark_nodes':nt.get('benchmark',0)}]); structure.to_csv(OUT_DIR/'network_structure.csv',index=False)


def avg_ht(Gc,probes=24,seed=0):
    if Gc.number_of_nodes()<2: return np.nan,np.nan
    nodes=list(Gc.nodes()); n=len(nodes); m=Gc.number_of_edges(); A=nx.to_scipy_sparse_array(Gc,nodelist=nodes,format='csc',dtype=np.float64); dv=np.asarray(A.sum(axis=1)).ravel(); L=sparse.diags(dv,format='csc')-A; lu=splu(L[:-1,:-1].tocsc()); rng=np.random.default_rng(seed); vals=[]
    for _ in range(probes):
        z=rng.choice(np.array([-1.0,1.0]),size=n); z=z-z.mean(); xg=lu.solve(z[:-1]); x=np.concatenate([xg,[0.0]]); x=x-x.mean(); vals.append(float(z@x))
    vals=np.asarray(vals); scale=2.0*m/(n-1); return scale*float(vals.mean()),scale*float(vals.std(ddof=1)/np.sqrt(len(vals)))

def sample_eff(Gc,sources=24,seed=0):
    n=Gc.number_of_nodes();
    if n<2: return np.nan,np.nan
    nodes=sorted(Gc.nodes()); rng=np.random.default_rng(seed); srcs=rng.choice(nodes,size=min(sources,n),replace=False); vals=[]
    for s in srcs:
        lens=nx.single_source_shortest_path_length(Gc,s); vals.append(sum(1.0/d for node,d in lens.items() if node!=s and d>0)/(n-1))
    vals=np.asarray(vals,float); return float(vals.mean()),float(vals.std(ddof=1)/np.sqrt(len(vals))) if len(vals)>1 else np.nan

BASE_HT,BASE_HT_SE=avg_ht(G0,HUTCHINSON_PROBES,stable_seed(MASTER_SEED,GRAPH_NAME,'base_ht')); BASE_EFF,BASE_EFF_SE=sample_eff(G0,GLOBAL_EFFICIENCY_SOURCES,stable_seed(MASTER_SEED,GRAPH_NAME,'base_eff'))
pd.DataFrame([{'graph':GRAPH_NAME,'baseline_hitting_time':BASE_HT,'hutchinson_se':BASE_HT_SE,'probes':HUTCHINSON_PROBES}]).to_csv(OUT_DIR/'baseline_transport.csv',index=False)
pd.DataFrame([{'graph':GRAPH_NAME,'baseline_global_efficiency':BASE_EFF,'source_sampling_se':BASE_EFF_SE,'sources':min(GLOBAL_EFFICIENCY_SOURCES,G0.number_of_nodes())}]).to_csv(OUT_DIR/'baseline_global_efficiency.csv',index=False)

base_deg=dict(G0.degree()); base_edges=list(G0.edges())
def pref_order(seed):
    rng=np.random.default_rng(seed); rec=[]
    for e in base_edges:
        u,v=e; rate=float(base_deg[u]+base_deg[v]); rec.append((rng.exponential(scale=1.0/max(rate,1e-12)),e))
    rec.sort(key=lambda x:x[0]); return [e for _,e in rec]
def rand_order(seed):
    rng=np.random.default_rng(seed); idx=rng.permutation(len(base_edges)); return [base_edges[i] for i in idx]
def damaged_lcc(order,frac):
    H=G0.copy(); H.remove_edges_from(order[:int(round(frac*len(order)))])
    if H.number_of_edges()==0: return None
    comps=list(nx.connected_components(H)); return H.subgraph(max(comps,key=len)).copy() if comps else None

def size_control(target_n,seed):
    if target_n>=G0.number_of_nodes(): return G0.copy()
    rng=np.random.default_rng(seed); nodes=list(G0.nodes()); start=nodes[int(rng.integers(0,len(nodes)))]; sel={start}; frontier=set(G0.neighbors(start))
    while len(sel)<target_n:
        if frontier:
            ch=list(frontier); nxt=ch[int(rng.integers(0,len(ch)))]; frontier.discard(nxt)
        else:
            cand=[x for x in nodes if x not in sel and any(y in sel for y in G0.neighbors(x))]
            if not cand: return None
            nxt=cand[int(rng.integers(0,len(cand)))]
        if nxt in sel: continue
        sel.add(nxt)
        for nb in G0.neighbors(nxt):
            if nb not in sel: frontier.add(nb)
    H=G0.subgraph(sel).copy();
    if not nx.is_connected(H): raise RuntimeError('Size control disconnected.')
    return H

raw_path=CHECKPOINT_DIR/'ai_domain_raw_checkpoint.csv'
if raw_path.exists(): raw=pd.read_csv(raw_path); done=set(raw['trial'].astype(int).unique()); rows=raw.to_dict('records'); print('Resuming',len(done),'trials')
else: done=set(); rows=[]
for trial in range(N_TRIALS):
    if trial in done: continue
    po=pref_order(stable_seed(MASTER_SEED,GRAPH_NAME,trial,'pref')); ro=rand_order(stable_seed(MASTER_SEED,GRAPH_NAME,trial,'rand'))
    for dtype,order in [('preferential',po),('random',ro)]:
        for frac in DAMAGE_FRACTIONS:
            H=damaged_lcc(order,float(frac))
            if H is None or H.number_of_nodes()<2: continue
            ht,se=avg_ht(H,HUTCHINSON_PROBES,stable_seed(MASTER_SEED,trial,dtype,frac,'ht')); eff,ese=sample_eff(H,GLOBAL_EFFICIENCY_SOURCES,stable_seed(MASTER_SEED,trial,dtype,frac,'eff'))
            rows.append({'graph':GRAPH_NAME,'trial':trial,'damage_type':dtype,'damage_fraction':float(frac),'lcc_nodes':H.number_of_nodes(),'lcc_edges':H.number_of_edges(),'lcc_fraction':H.number_of_nodes()/G0.number_of_nodes(),'hitting_time':ht,'hutchinson_se':se,'relative_hitting_time':ht/BASE_HT,'global_efficiency':eff,'global_efficiency_se':ese,'relative_global_efficiency':eff/BASE_EFF})
    pd.DataFrame(rows).to_csv(raw_path,index=False); print('Completed trial',trial+1,'/',N_TRIALS)
raw=pd.DataFrame(rows); raw.to_csv(OUT_DIR/'ai_domain_raw.csv',index=False)

curve=[]
for (dtype,frac),g in raw.groupby(['damage_type','damage_fraction']):
    rm,rlo,rhi=t_ci(g.relative_hitting_time); em,elo,ehi=t_ci(g.relative_global_efficiency); lm,llo,lhi=t_ci(g.lcc_fraction)
    curve.append({'graph':GRAPH_NAME,'damage_type':dtype,'damage_fraction':frac,'n_trials':g.trial.nunique(),'mean_relative_hitting_time':rm,'ht_ci95_low':rlo,'ht_ci95_high':rhi,'mean_relative_global_efficiency':em,'eff_ci95_low':elo,'eff_ci95_high':ehi,'mean_lcc_fraction':lm,'lcc_ci95_low':llo,'lcc_ci95_high':lhi})
curve=pd.DataFrame(curve); curve.to_csv(OUT_DIR/'curve_statistics.csv',index=False)

scores=[]
for (trial,dtype),g in raw.groupby(['trial','damage_type']):
    g=g.sort_values('damage_fraction'); rel=g.relative_hitting_time.to_numpy(float); reff=g.relative_global_efficiency.to_numpy(float)
    scores.append({'graph':GRAPH_NAME,'trial':int(trial),'damage_type':dtype,'sustained_af':float(np.mean(np.maximum(0,1-rel))),'sustained_global_efficiency_improvement':float(np.mean(np.maximum(0,reff-1)))})
scores=pd.DataFrame(scores); scores.to_csv(OUT_DIR/'trial_sustained_scores.csv',index=False)
p=scores[scores.damage_type=='preferential'][['trial','sustained_af']].rename(columns={'sustained_af':'pref'}); r=scores[scores.damage_type=='random'][['trial','sustained_af']].rename(columns={'sustained_af':'random'}); pair=p.merge(r,on='trial'); delta=pair.pref-pair.random; pr_stat,pr_p=safe_wilcoxon(delta,'greater'); pr_rb=matched_rb(delta); dm,dlo,dhi=t_ci(delta)
pd.DataFrame([{'graph':GRAPH_NAME,'n_trials':len(pair),'mean_sustained_preferential':pair.pref.mean(),'mean_sustained_random':pair.random.mean(),'mean_difference_pref_minus_random':dm,'difference_ci95_low':dlo,'difference_ci95_high':dhi,'wilcoxon_stat':pr_stat,'p_one_sided_pref_gt_random':pr_p,'rank_biserial_effect':pr_rb}]).to_csv(OUT_DIR/'preferential_vs_random.csv',index=False)

srows=[]
for trial in range(N_TRIALS):
    po=pref_order(stable_seed(MASTER_SEED,GRAPH_NAME,trial,'pref'))
    for frac in SIZE_CONTROL_FRACTIONS:
        Hp=damaged_lcc(po,float(frac)); Hs=size_control(Hp.number_of_nodes(),stable_seed(MASTER_SEED,trial,frac,'size'))
        hp,_=avg_ht(Hp,HUTCHINSON_PROBES,stable_seed(MASTER_SEED,trial,frac,'hp')); hs,_=avg_ht(Hs,HUTCHINSON_PROBES,stable_seed(MASTER_SEED,trial,frac,'hs')); ep,_=sample_eff(Hp,GLOBAL_EFFICIENCY_SOURCES,stable_seed(MASTER_SEED,trial,frac,'ep')); es,_=sample_eff(Hs,GLOBAL_EFFICIENCY_SOURCES,stable_seed(MASTER_SEED,trial,frac,'es'))
        srows.append({'graph':GRAPH_NAME,'trial':trial,'damage_fraction':float(frac),'pref_nodes':Hp.number_of_nodes(),'size_nodes':Hs.number_of_nodes(),'pref_hitting_time':hp,'size_hitting_time':hs,'pref_over_size_ht':hp/hs,'pref_global_efficiency':ep,'size_global_efficiency':es,'pref_over_size_efficiency':ep/es})
size_pair=pd.DataFrame(srows); size_pair.to_csv(OUT_DIR/'size_matched_pair_level.csv',index=False)
trial_size=size_pair.groupby('trial',as_index=False).agg(pref_over_size_ht=('pref_over_size_ht','mean'),pref_over_size_efficiency=('pref_over_size_efficiency','mean')); trial_size.to_csv(OUT_DIR/'size_matched_trial_level.csv',index=False)
htdiff=1-trial_size.pref_over_size_ht.to_numpy(float); ht_stat,ht_p=safe_wilcoxon(htdiff,'greater'); ht_rb=matched_rb(htdiff); ht_mean,ht_lo,ht_hi=t_ci(trial_size.pref_over_size_ht)
ediff=trial_size.pref_over_size_efficiency.to_numpy(float)-1; e_stat,e_p=safe_wilcoxon(ediff,'greater'); e_rb=matched_rb(ediff); e_mean,e_lo,e_hi=t_ci(trial_size.pref_over_size_efficiency)
pd.DataFrame([{'graph':GRAPH_NAME,'n_independent_trials':len(trial_size),'mean_pref_over_size_ht':ht_mean,'ht_ratio_ci95_low':ht_lo,'ht_ratio_ci95_high':ht_hi,'ht_wilcoxon_stat':ht_stat,'ht_p_one_sided_pref_lt_size':ht_p,'ht_rank_biserial_benefit':ht_rb,'mean_pref_over_size_efficiency':e_mean,'eff_ratio_ci95_low':e_lo,'eff_ratio_ci95_high':e_hi,'eff_wilcoxon_stat':e_stat,'eff_p_one_sided_pref_gt_size':e_p,'eff_rank_biserial_benefit':e_rb}]).to_csv(OUT_DIR/'size_matched_controls.csv',index=False)

pc=curve[curve.damage_type=='preferential'].copy(); pc['mean_improvement_pct']=100*(1-pc.mean_relative_hitting_time); best=pc.loc[pc.mean_improvement_pct.idxmax()]; levels=int((pc.mean_relative_hitting_time<IMPROVEMENT_THRESHOLD).sum()); sap=float(scores[scores.damage_type=='preferential'].sustained_af.mean()); sar=float(scores[scores.damage_type=='random'].sustained_af.mean()); sustained_ok=levels>=MIN_IMPROVEMENT_LEVELS; lcc_ok=float(best.mean_lcc_fraction)>=MIN_LCC_FRACTION; pvr_ok=np.isfinite(pr_p) and pr_p<ALPHA; size_ok=np.isfinite(ht_p) and ht_p<ALPHA and ht_mean<1
if sustained_ok and lcc_ok and pvr_ok and size_ok: classification='Controlled antifragile transport'
elif sustained_ok and lcc_ok and (pvr_ok or size_ok): classification='Partial / control-dependent evidence'
else: classification='Unsupported under strict controls'
summary=pd.DataFrame([{'graph':GRAPH_NAME,'domain':'observed LLM agent/tool execution traces','trials':N_TRIALS,'max_mean_ht_improvement_pct':float(best.mean_improvement_pct),'best_damage_fraction':float(best.damage_fraction),'damage_levels_ge_2pct':levels,'mean_sustained_af_preferential':sap,'mean_sustained_af_random':sar,'pref_vs_random_p':pr_p,'pref_vs_random_rank_biserial':pr_rb,'mean_lcc_fraction_at_best':float(best.mean_lcc_fraction),'mean_pref_over_size_ht':ht_mean,'size_control_p_trial_level':ht_p,'size_control_rank_biserial':ht_rb,'mean_pref_over_size_global_efficiency':e_mean,'global_eff_size_control_p_trial_level':e_p,'classification':classification}]); summary.to_csv(OUT_DIR/'network_summary.csv',index=False)

# PNG-only figures
for dtype,label in [('preferential','Preferential'),('random','Random')]:
    sub=curve[curve.damage_type==dtype].sort_values('damage_fraction'); x=100*sub.damage_fraction.to_numpy(float); y=sub.mean_relative_hitting_time.to_numpy(float); lo=sub.ht_ci95_low.to_numpy(float); hi=sub.ht_ci95_high.to_numpy(float); plt.plot(x,y,marker='o',label=label); plt.fill_between(x,lo,hi,alpha=.20)
plt.axhline(1,ls='--',lw=1); plt.xlabel('Edge Damage Fraction (%)'); plt.ylabel('Relative Average Hitting Time'); plt.title('Exgentic Agent/Tool Graph: Preferential vs Random Damage'); plt.legend(); plt.tight_layout(); save_png('exgentic_agenttool_preferential_vs_random')
for dtype,label in [('preferential','Preferential'),('random','Random')]:
    sub=curve[curve.damage_type==dtype].sort_values('damage_fraction'); plt.plot(100*sub.damage_fraction,sub.mean_lcc_fraction,marker='o',label=label)
plt.xlabel('Edge Damage Fraction (%)'); plt.ylabel('Fraction of Baseline Nodes in LCC'); plt.title('Exgentic Agent/Tool Graph: LCC Retention'); plt.legend(); plt.tight_layout(); save_png('exgentic_agenttool_lcc_retention')
vals=[size_pair[size_pair.damage_fraction==f].pref_over_size_ht.to_numpy(float) for f in SIZE_CONTROL_FRACTIONS]; plt.figure(figsize=(6.5,4.4)); plt.boxplot(vals,tick_labels=[f'{int(f*100)}%' for f in SIZE_CONTROL_FRACTIONS]); plt.axhline(1,ls='--',lw=1); plt.xlabel('Preferential Edge Damage'); plt.ylabel('Preferential HT / Size-Matched HT'); plt.title('Exgentic Agent/Tool Graph: Size-Matched Hitting-Time Control'); plt.tight_layout(); save_png('exgentic_agenttool_size_matched_ht')
plt.figure(figsize=(6.5,4.4))
for dtype,label in [('preferential','Preferential'),('random','Random')]:
    sub=curve[curve.damage_type==dtype].sort_values('damage_fraction'); x=100*sub.damage_fraction.to_numpy(float); y=sub.mean_relative_global_efficiency.to_numpy(float); lo=sub.eff_ci95_low.to_numpy(float); hi=sub.eff_ci95_high.to_numpy(float); plt.plot(x,y,marker='o',label=label); plt.fill_between(x,lo,hi,alpha=.20)
plt.axhline(1,ls='--',lw=1); plt.xlabel('Edge Damage Fraction (%)'); plt.ylabel('Relative Global Efficiency'); plt.title('Exgentic Agent/Tool Graph: Global Efficiency'); plt.legend(); plt.tight_layout(); save_png('exgentic_agenttool_global_efficiency_pref_vs_random')
vals=[size_pair[size_pair.damage_fraction==f].pref_over_size_efficiency.to_numpy(float) for f in SIZE_CONTROL_FRACTIONS]; plt.figure(figsize=(6.5,4.4)); plt.boxplot(vals,tick_labels=[f'{int(f*100)}%' for f in SIZE_CONTROL_FRACTIONS]); plt.axhline(1,ls='--',lw=1); plt.xlabel('Preferential Edge Damage'); plt.ylabel('Preferential Efficiency / Size-Matched Efficiency'); plt.title('Exgentic Agent/Tool Graph: Global-Efficiency Size Control'); plt.tight_layout(); save_png('exgentic_agenttool_global_efficiency_size_control')
plt.figure(figsize=(6.5,4.4)); labels=list(nt.keys()); counts=[nt[x] for x in labels]; plt.bar(labels,counts); plt.ylabel('Nodes in Baseline LCC'); plt.title('Exgentic Agent/Tool Graph: Observed Node-Type Composition'); plt.xticks(rotation=20); plt.tight_layout(); save_png('exgentic_agenttool_node_type_composition')

# LaTeX tables
fmtp=lambda x: '--' if not np.isfinite(x) else (f'{x:.2e}' if x<.001 else f'{x:.3f}')
(OUT_DIR/'domain_validation_table.tex').write_text('\n'.join([r'\begin{table}[!t]',r'\centering',r'\caption{Domain-specific validation results for the observed Exgentic agent/tool dependency graph.}',r'\label{tab:exgentic_validation}',r'\footnotesize',r'\begin{tabular}{lr}',r'\toprule',r'Metric & Result \\',r'\midrule',f'Nodes & {G0.number_of_nodes():,} \\',f'Edges & {G0.number_of_edges():,} \\',f'Modularity & {modularity:.3f} \\',f'Maximum HT improvement & {float(best.mean_improvement_pct):.2f}\\% \\',f'Best damage level & {100*float(best.damage_fraction):.0f}\\% \\',f'Preferential sustained $A_{{AF}}$ & {sap:.4f} \\',f'Random sustained $A_{{AF}}$ & {sar:.4f} \\',f'$p_{{PR}}$ & {fmtp(pr_p)} \\',f'LCC retention at best & {float(best.mean_lcc_fraction):.3f} \\',f'$H_{{pref}}/H_{{size}}$ & {ht_mean:.3f} \\',f'$p_{{size}}$ (trial level) & {fmtp(ht_p)} \\',f'$E_{{pref}}/E_{{size}}$ & {e_mean:.3f} \\',f'$p_{{E,size}}$ (trial level) & {fmtp(e_p)} \\',r'\bottomrule',r'\end{tabular}',r'\end{table}']),encoding='utf-8')
(OUT_DIR/'domain_structure_table.tex').write_text('\n'.join([r'\begin{table}[!t]',r'\centering',r'\caption{Observed structural characteristics of the Exgentic agent/tool dependency graph.}',r'\label{tab:exgentic_structure}',r'\footnotesize',r'\begin{tabular}{lr}',r'\toprule',r'Characteristic & Value \\',r'\midrule',f'Nodes & {G0.number_of_nodes():,} \\',f'Edges & {G0.number_of_edges():,} \\',f'Mean degree & {mean_degree:.3f} \\',f'Degree CV & {degree_cv:.3f} \\',f'Average clustering & {nx.average_clustering(G0):.3f} \\',f'Transitivity & {nx.transitivity(G0):.3f} \\',f'Modularity & {modularity:.3f} \\',f'Communities & {len(communities)} \\',f'Tool nodes & {nt.get("tool",0)} \\',f'Model nodes & {nt.get("model",0)} \\',f'Harness nodes & {nt.get("harness",0)} \\',f'Benchmark nodes & {nt.get("benchmark",0)} \\',r'\bottomrule',r'\end{tabular}',r'\end{table}']),encoding='utf-8')

summary_txt=f'''DOMAIN-SPECIFIC AI SECURITY VALIDATION — EXGENTIC AGENT/TOOL GRAPH\n\nDataset: {DATASET_ID}\nURL: {DATASET_URL}\n\nTraces processed: {trace_count:,}\nSpans processed: {span_count:,}\nObserved tool calls: {tool_call_count:,}\nBaseline LCC: {G0.number_of_nodes():,} nodes, {G0.number_of_edges():,} edges\n\nPrimary Markovian result:\n  max mean improvement: {float(best.mean_improvement_pct):.4f}%\n  best damage fraction: {float(best.damage_fraction):.2f}\n  levels >=2% improvement: {levels}/9\n  sustained preferential AF: {sap:.6f}\n  sustained random AF: {sar:.6f}\n  preferential vs random p: {pr_p:.10g}\n  rank-biserial: {pr_rb:.6f}\n  LCC retention at best: {float(best.mean_lcc_fraction):.6f}\n\nIndependent-trial size control:\n  H_pref/H_size: {ht_mean:.6f}\n  95% CI: [{ht_lo:.6f}, {ht_hi:.6f}]\n  one-sided p: {ht_p:.10g}\n  rank-biserial benefit: {ht_rb:.6f}\n\nConfirmatory global efficiency:\n  E_pref/E_size: {e_mean:.6f}\n  95% CI: [{e_lo:.6f}, {e_hi:.6f}]\n  one-sided p: {e_p:.10g}\n  rank-biserial benefit: {e_rb:.6f}\n\nClassification: {classification}\n\nIMPORTANT: This result has NOT been merged into the manuscript. Review first, then update the paper.\n'''
(OUT_DIR/'RESULTS_SUMMARY.txt').write_text(summary_txt,encoding='utf-8')

required_outputs=[OUT_DIR/'dataset_provenance.csv',OUT_DIR/'network_structure.csv',OUT_DIR/'baseline_transport.csv',OUT_DIR/'baseline_global_efficiency.csv',OUT_DIR/'ai_domain_raw.csv',OUT_DIR/'curve_statistics.csv',OUT_DIR/'trial_sustained_scores.csv',OUT_DIR/'preferential_vs_random.csv',OUT_DIR/'size_matched_pair_level.csv',OUT_DIR/'size_matched_trial_level.csv',OUT_DIR/'size_matched_controls.csv',OUT_DIR/'network_summary.csv',OUT_DIR/'domain_validation_table.tex',OUT_DIR/'domain_structure_table.tex',OUT_DIR/'RESULTS_SUMMARY.txt',FIG_DIR/'exgentic_agenttool_preferential_vs_random.png',FIG_DIR/'exgentic_agenttool_lcc_retention.png',FIG_DIR/'exgentic_agenttool_size_matched_ht.png',FIG_DIR/'exgentic_agenttool_global_efficiency_pref_vs_random.png',FIG_DIR/'exgentic_agenttool_global_efficiency_size_control.png',FIG_DIR/'exgentic_agenttool_node_type_composition.png']
missing=[str(p) for p in required_outputs if not p.exists()]; pdfs=list(OUT_DIR.rglob('*.pdf'))
print('\n'+'='*88+'\nFINAL DOMAIN-VALIDATION AUDIT\n'+'='*88); print(summary_txt); print('Missing outputs:',len(missing)); print('PDF files created:',len(pdfs))
if missing: raise RuntimeError('Missing outputs: '+str(missing))
if pdfs: raise RuntimeError('PNG-only requirement violated: '+str(pdfs))
print('SUCCESS — all figures are PNG only.'); print('Review:',OUT_DIR/'RESULTS_SUMMARY.txt'); print('Figures:',FIG_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
1. LOADING REAL AGENT EXECUTION TRACES
Exgentic/agent-llm-traces-v2
500 traces | 292 nodes | 1,742 edges | 13,414 tool calls
1,000 traces | 347 nodes | 2,454 edges | 23,475 tool calls
1,500 traces | 386 nodes | 2,964 edges | 41,689 tool calls
2,000 traces | 400 nodes | 3,025 edges | 54,328 tool calls
2,500 traces | 400 nodes | 3,036 edges | 60,846 tool calls
3,000 traces | 400 nodes | 3,040 edges | 62,513 tool calls
3,500 traces | 404 nodes | 3,053 edges | 74,455 tool calls
4,000 traces | 408 nodes | 3,098 edges | 100,180 tool calls
4,500 traces | 408 nodes | 3,104 edges | 112,632 tool calls
5,000 traces | 408 nodes | 3,105 edges | 119,449 tool calls
5,500 traces | 426 nodes | 3,167 edges | 134,288 tool calls
6,000 traces | 440 nodes | 3,319 edges | 137,770 tool calls
6,500 traces | 453 nodes | 3,430 edges | 140,877 tool calls
7,000 traces | 464 nodes | 3,551